# Weight-Space Identity Research, Adapted to Open-Weight ~2B Models
### Qwen2.5-1.5B &nbsp;|&nbsp; Gemma-2-2b &nbsp;|&nbsp; Llama-3.2-1B

**Adaptation note.** This notebook re-runs the permutation/rotation "identity in weights
vs. connectivity" research program from the original GPT-2 notebook, generalized to
three real open-weight model families. It was written and syntax-checked in an
environment with no internet/GPU access, so the numeric outputs below have **not** been
executed by the assistant that wrote it -- every cell is meant to be run top-to-bottom in
Colab, and every experiment carries its own numeric self-check (e.g. "max abs logit
diff, expect ~1e-5") so you can verify correctness directly from real output rather than
trusting a description. Nothing here uses synthetic weights or fabricated numbers --
these are the actual pretrained checkpoints.

**Why this isn't a mechanical copy of the GPT-2 version.** GPT-2 uses `Conv1D` layers
($y = xW + b$, permute output *columns*) and plain multi-head attention (num query heads
== num KV heads). Llama/Qwen/Gemma use `nn.Linear` ($y = xW^\top + b$, permute output
*rows*), **SwiGLU/GeGLU** MLPs (two parallel projections, `gate_proj` and `up_proj`,
multiplied elementwise before `down_proj` -- GPT-2 has only one projection, `c_fc`), and
**grouped-query attention / GQA** (fewer KV heads than query heads, so head permutation
has to respect group membership -- GPT-2 has none of this). Every transformation
function below is re-derived for these structures, not copy-pasted with renamed
variables.

**Models.**

| Family | Checkpoint | Params | MLP | Attention |
|---|---|---|---|---|
| Qwen | `Qwen/Qwen2.5-1.5B` | 1.5B | SwiGLU (SiLU) | GQA |
| Gemma | `google/gemma-2-2b` | 2.6B | GeGLU (gelu_tanh) | GQA + sliding-window on alternating layers |
| Llama | `meta-llama/Llama-3.2-1B` | 1.24B | SwiGLU (SiLU) | GQA |

Gemma-2-2b and Llama-3.2-1B are gated on the Hugging Face Hub -- accept the license on
the model page, then run `huggingface_hub.login()` in Colab before loading.

**v3 fix: dtype mismatches under bf16/fp16.** Running this on a GPU where
`PRECISION_MODE='auto'` selects bf16 (e.g. a 16GB Kaggle T4 holding all three
checkpoints) surfaced two related bugs, both now fixed:
1. `build_block_permutation` (used by the GQA attention-head permutation in
   Section 2) built its permutation matrix in the default float32 while the
   model's Q/K/V/O weights were bf16 -- `P.T @ attn.q_proj.weight.data` then
   fails with `RuntimeError: expected mat1 and mat2 to have the same dtype`.
   It now takes an explicit `dtype` argument, and every call site passes the
   attention block's actual weight dtype.
2. `make_local_orthogonal_mlp`, `make_global_rotation`, and
   `rotation_angle_sweep_mlp` (Section 3 / Part 4) built their random matrix
   directly in the model's bf16/fp16 dtype and called `torch.linalg.qr` on it --
   but cuSOLVER (the CUDA backend for QR) has no half-precision kernel, so this
   would have failed the moment execution reached those cells, with something
   like `"geqrf_cuda" not implemented for 'BFloat16'`. Both now build the random
   matrix and run QR in float32 (always supported), then cast only the final
   orthogonal `Q` to the model's dtype right before it's used in a matmul against
   model weights.


---

## Running this on Kaggle (memory + auth notes)

**Three checkpoints, one GPU.** Qwen2.5-1.5B + Gemma-2-2b + Llama-3.2-1B add up to
~5.3B params. Loaded naively in float32 *and* kept resident for the whole notebook
(needed, since Part 3's cross-model comparisons reuse them at the end), that alone is
~21GB -- more than a single Kaggle T4 (16GB) or P100 (16GB) has. On top of that, the
*naive* way to test a weight perturbation is `copy.deepcopy(model)`, which duplicates
every one of a model's parameters just to change a handful of tensors in one layer --
across this notebook's ~90+ perturbation calls (multi-seed sweeps, boundary sweeps,
perturbation controls, rotation sweeps) that pattern would repeatedly spike GPU memory
far past what any single Kaggle GPU offers.

This version fixes that at the source rather than patching around it:
- **`ConditionModel`** (new cell right after `MODEL_CONFIGS`) represents a perturbed
  variant (e.g. "aligned permutation of layer 4's MLP") as a *cheap view* over the one
  already-loaded base model: it stores only the small tensors that actually change,
  swaps them onto the live model immediately before a forward pass, and swaps the
  originals back immediately after. No full-model copy is ever made for these.
- **Sequential, size-aware loading**: models load one at a time with
  `low_cpu_mem_usage=True`, and precision (fp32 vs. bf16/fp16) is chosen automatically
  based on whether all three checkpoints will actually fit on the detected GPU (see
  `PRECISION_MODE` in Part 0). Lower precision means the "exact symmetry" numeric
  checks throughout will show small nonzero diffs (bf16/fp16 floating-point noise)
  rather than fp32's near-machine-epsilon values -- expected, and still confirms the
  same qualitative conclusion (aligned ~0, unaligned large).
- **`free_memory()` / `print_gpu_mem()`** calls after every model load and after the
  few remaining genuinely-full-size operations (global residual-stream rotation,
  fine-tuning-preserves-symmetry check) that can't avoid touching the whole model.

**Hugging Face auth for the gated checkpoints (Gemma-2-2b, Llama-3.2-1B).** Don't paste
a raw token into a notebook cell -- notebooks get saved, re-shared, and (on Kaggle)
sometimes made public, and a leaked **write**-scoped token can modify or delete things
on your account. Instead: open this notebook's **Add-ons -> Secrets**, add a secret
named `HF_TOKEN` with your token as the value, toggle it on for this notebook, and the
loader cell below will pick it up automatically (with an `HF_TOKEN` environment
variable as a non-Kaggle fallback). If you pasted a token anywhere else while setting
this up, it's worth rotating/revoking it from your HF account settings afterward.


**v4 additions (this revision): closing the two gaps in the reviewer's Points 3-5.**
1. **Real DTYPE/PRECISION_MODE bug fix.** `DTYPE` and `PRECISION_MODE` were referenced
   throughout (the "v3 fix" note above, the loader cell, its closing print statement) but
   never actually assigned -- the loader would have raised `NameError` on first real
   execution. Now defined with real, GPU-capability-aware logic in Part 0.
2. **Point 5's ANOVA now runs on real data.** The inter-model ANOVA / Bonferroni
   pairwise tests previously ran on `np.random.normal(mean, sd)` samples resampled from a
   reported confidence interval. It now uses the actual N=15 per-seed arrays already
   computed at STEP 8 (`seed_sweep_records`), same for the effect-size table just above it.
3. **Point 4's second named mechanism, previously untested.** The reviewer's Point 4 named
   two candidate mechanisms -- redundant circuits (tested) and a low-effective-rank readout
   (not tested until now). A new "Further experiment C" isolates rank from redundancy
   directly (orthogonal-factor, no-duplicate-column construction at controlled numerical
   rank) and reports each rank's actual effective rank via its singular-value spectrum's
   participation ratio, not just the constructor's intended rank.
4. **HPC instrumentation on both PENDING-GPU real-checkpoint cells** (Point 3 and Point 4
   follow-ups): a shared autocast context matching each checkpoint's own load dtype, plus
   per-model wall-clock timing, `print_gpu_mem()`, and `free_memory()` calls, so peak memory
   and per-model cost are visible rather than assumed once these run on a real GPU.

**v5 additions: two more real bugs, and toy checks #1/#2 replaced with real cross-model checks.**
1. **Real device-placement bug, confirmed by an actual Kaggle T4x2 run.** `make_aligned_mlp_permutation_real`
   accepted a `device` argument but never used it -- its returned deep copy stayed on CPU while
   `finetune_in_place` moved only the training token ids to `DEVICE`, crashing the very first
   training step of Part 3 Step C's aligned-copy branch with a device-mismatch `RuntimeError`.
   Fixed at the source: the function now moves its own returned copy to `device` before returning.
2. **Toy check #1 (RMSNorm sandwich) and toy check #2 (dilution ratio) replaced with Real Check #1
   and Real Check #2.** Both previously ran on random Gaussian weights in a small synthetic MLP,
   which could support the post-norm/dilution hypotheses without ever showing whether either holds
   on the actual checkpoints, or generalizes across architectures rather than being an artifact of
   whichever one motivated it. Real Check #1 causally bypasses Gemma-2's actual, trained
   `post_feedforward_layernorm` submodule (forward temporarily replaced with identity) and compares
   against its native behavior and against Qwen/Llama's native behavior (neither has an equivalent
   submodule). Real Check #2 measures the branch/residual dilution ratio and downstream cosine
   similarity at every rotation angle, on real hooked activations, for all three checkpoints
   simultaneously -- directly testing whether the same ratio-predicts-invariance relationship holds
   across architectures or is model-specific. The now-redundant baseline-only branch/residual-ratio
   probe in the Point 3 PENDING-GPU follow-up cell was removed in favor of Real Check #2's full
   sweep, rather than computing the same quantity twice.

**v6 addition: real Kaggle run reached the Point 4 CKA/attention-entropy cell and crashed there.**
`attention_entropy_batched` called `model(..., output_attentions=True)` on checkpoints loaded with
the default `sdpa` attention implementation, which silently sets `out.attentions = None` instead of
raising -- the crash (`TypeError: 'NoneType' object is not subscriptable`) surfaced three frames
away from the actual cause. Fixed with a small `eager_attention(model)` context manager that
switches the model to `eager` attention (which materializes real per-head attention weights) only
for the duration of this specific call, then restores the original implementation -- not a
permanent switch for the rest of the notebook, since eager is slower and higher-memory than sdpa.
Everything through Part 4's rotation sweep, Real Check #1, Real Check #2, Further experiment A, and
Point 3 Follow-up B ran cleanly on real weights before this -- see the changelog entries above for
what those numbers showed.


> **Operating question for this notebook.** Under which structural transformations of a
> network's weights does its input-output function persist -- exactly, approximately, or not
> at all -- and which properties of the architecture predict where that boundary sits? Every
> section below is a measurement toward that question. Where the results turn out to support a
> broader vocabulary for describing what's being measured, that discussion is at the very end
> (after Part 6), built from the specific numbers obtained here rather than assumed in advance.


## Part 0 -- Setup

**Motivating literature (unchanged from the original notebook).** The permutation-symmetry
claim traces to **Hecht-Nielsen (1990)**, revived by **Entezari, Sedghi, Saukh & Neyshabur
(2022)** and **Ainsworth, Hayase & Srinivasa (2023)**. Everything below is the same
falsification-test design applied to three new architectures instead of one.

In [ ]:

import os
# Reduces CUDA allocator fragmentation across the many small alloc/free cycles
# this notebook does (must be set before torch initializes CUDA).
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import copy, math, json, gc
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from scipy.linalg import orthogonal_procrustes
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

# ---------------------------------------------------------------------------
# Memory-safety helpers, used throughout the notebook after anything that
# touches GPU memory in a non-trivial way (model loads, whole-model rotations,
# fine-tuning copies). Cheap per-layer perturbations (see ConditionModel,
# defined right after MODEL_CONFIGS below) mostly don't need these, since they
# never allocate a whole extra model in the first place.
# ---------------------------------------------------------------------------
def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def print_gpu_mem(tag=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        label = f" [{tag}]" if tag else ""
        print(f"  GPU mem{label}: allocated={alloc:.2f}GB  reserved={reserved:.2f}GB  total={total:.2f}GB")

# ---------------------------------------------------------------------------
# HPC note (fixes the PART 4 OOM from the previous run): every random-
# orthogonal-matrix construction in this notebook (local MLP rotation, global
# residual rotation, the PART 4 rotation-angle sweep) needs a `dim x dim`
# fp32 QR decomposition, and cuSOLVER's QR kernel only supports fp32/fp64 --
# so building it directly on `DEVICE` forces a full fp32 d_ff x d_ff (or
# d_model x d_model) buffer onto the single busiest, most memory-constrained
# device in the whole notebook, on top of three already-resident checkpoints.
# CPU RAM is far more abundant and is untouched by everything else here, so
# `cpu_orthogonal` builds and QR-decomposes ENTIRELY on CPU; callers move only
# the final matrix (cast down to the model's dtype, which is also half the
# fp32 size) to GPU, right before the one matmul that needs it, for as short
# a time as possible.
# ---------------------------------------------------------------------------
def cpu_orthogonal(dim, seed=0):
    """Random `dim x dim` orthogonal matrix, built and QR-decomposed on CPU.

    Returns a CPU float32 tensor. Cast + .to(device) only the matrix you still
    need, right before consuming it -- never hold the fp32 CPU copy and a GPU
    copy alive at the same time (see make_local_orthogonal_mlp,
    make_global_rotation, rotation_angle_sweep_mlp for the pattern). Uses its
    own CPU generator instead of the global RNG (torch.manual_seed) so it has
    no side effect on unrelated random draws elsewhere in the notebook.
    """
    gen = torch.Generator(device='cpu').manual_seed(seed)
    A = torch.randn(dim, dim, dtype=torch.float32, generator=gen)
    Q, _ = torch.linalg.qr(A)
    del A
    return Q

import contextlib

@contextlib.contextmanager
def cuda_oom_guard(label=""):
    """Wrap a GPU-memory-heavy block so an OOM prints actionable diagnostics
    and frees the cache before propagating -- instead of a bare traceback
    that papermill/Kaggle turns into a full from-scratch re-run. Does NOT
    swallow the error: an OOM here means a real memory bug upstream (an extra
    full-size buffer landed on GPU, or a condition wasn't freed before the
    next one was built) that should be fixed at the source, not silently
    retried into a hang.
    """
    try:
        yield
    except torch.cuda.OutOfMemoryError:
        print(f"\n  !! CUDA OOM in {label or '(unlabeled block)'} !!")
        print_gpu_mem("at OOM")
        free_memory()
        print_gpu_mem("after emergency free_memory()")
        raise

# ---------------------------------------------------------------------------
# Precision selection. 'auto' picks float32 if all checkpoints comfortably fit
# on the detected GPU, otherwise drops to bf16 (or fp16 if bf16 isn't
# supported) so the three checkpoints and their transient perturbations don't
# exceed available VRAM. Force 'fp32' / 'bf16' / 'fp16' if you want to
# override the heuristic.
# ---------------------------------------------------------------------------
PRECISION_MODE = 'auto'  # 'auto' | 'fp32' | 'bf16' | 'fp16'

# Rough param counts for the three default checkpoints, used only for the
# 'auto' fit estimate below -- update if you change MODEL_CONFIGS to
# different-sized checkpoints.
_EST_TOTAL_PARAMS = 1.54e9 + 2.61e9 + 1.24e9  # Qwen2.5-1.5B + Gemma-2-2b + Llama-3.2-1B

def pick_dtype():
    if DEVICE != 'cuda':
        return torch.float32, 'float32 (CPU run)'
    if PRECISION_MODE == 'fp32':
        return torch.float32, 'float32 (forced)'
    if PRECISION_MODE == 'bf16':
        return torch.bfloat16, 'bfloat16 (forced)'
    if PRECISION_MODE == 'fp16':
        return torch.float16, 'float16 (forced)'
    total_gpu = torch.cuda.get_device_properties(0).total_memory
    fp32_bytes = _EST_TOTAL_PARAMS * 4 * 1.3  # +30% headroom for activations/temp tensors
    if fp32_bytes < total_gpu * 0.85:
        return torch.float32, 'float32 (auto: fp32 fits comfortably on this GPU)'
    bf16_ok = torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16_ok else torch.float16
    name = 'bfloat16' if bf16_ok else 'float16'
    return dtype, f"{name} (auto: fp32 would not fit all three checkpoints on this GPU)"

DTYPE, DTYPE_REASON = pick_dtype()
print(f"Selected dtype: {DTYPE_REASON}")
if DTYPE != torch.float32:
    print("NOTE: the 'expect ~1e-5 / near-fp32-noise' exactness comments throughout this")
    print("      notebook should be read relative to this dtype -- bf16/fp16 exactness")
    print("      checks will show diffs on the order of 1e-2 to 1e-3, not fp32's ~1e-5,")
    print("      but the qualitative conclusion (aligned ~0, unaligned large) is unchanged.")

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25,
})
COLORS = {
    'baseline': '#2b2b2b', 'aligned': '#1f77b4', 'unaligned': '#d62728',
    'head_aligned': '#2ca02c', 'head_unaligned': '#ff7f0e',
    'rotation_local': '#9467bd', 'rotation_global': '#8c564b',
    'noise': '#e377c2', 'deletion': '#17becf',
    'Qwen2.5-1.5B': '#1f77b4', 'Gemma-2-2b': '#2ca02c', 'Llama-3.2-1B': '#d62728',
}


### Performance setup (added)

Three cheap, correctness-neutral flags for this hardware, plus one substantive change used by
all the *new* cells added below: `torch.inference_mode()` instead of `torch.no_grad()` for
inference-only passes (skips version-counter bookkeeping `no_grad` still does; strictly a
superset of `no_grad`'s guarantees, never a behavior change for code that doesn't try to
autograd-track its way back into the block afterward, which nothing here does). The existing
per-model offload/restore and `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` pattern already
in Part 0 is the right call for this notebook's actual bottleneck (peak GPU *memory* across three
co-resident ~1-3B models on a single T4), so it's left as-is rather than replaced.


In [ ]:
# Performance flags (safe on any CUDA device; no-ops / negligible cost if unsupported).
torch.backends.cudnn.benchmark = True          # let cuDNN autotune algo choice for this fixed-
                                                # input-size workload (harmless even with no
                                                # convs on the hot path; correct to leave on for
                                                # any incidental use, e.g. tokenizer backends).
torch.backends.cuda.matmul.allow_tf32 = True    # no-op on Turing (T4); free win if this ever
torch.backends.cudnn.allow_tf32 = True          # runs on Ampere+ (A100/A10/3090/4090/etc).
print("cudnn.benchmark:", torch.backends.cudnn.benchmark)
print("matmul.allow_tf32:", torch.backends.cuda.matmul.allow_tf32)


# ---------------------------------------------------------------------------
# BUG FIX (this revision): PRECISION_MODE and DTYPE are referenced throughout
# this notebook (the "v3 fix" note in Part 0, the loader cell's
# `from_pretrained(..., dtype=DTYPE, ...)` call, and its closing print
# statement) but neither was ever actually assigned -- the loader cell would
# raise `NameError: name 'DTYPE' is not defined` on first real execution.
# Fixed at the source, with the dtype chosen from the ACTUAL detected device
# rather than assumed:
#   - no CUDA -> float32 (numerics matter more than memory on CPU, and CPU
#     gets no speedup from half precision anyway).
#   - CUDA, compute capability < 8 (Turing: T4, P100 -- Kaggle's two GPU
#     options) -> float16, since Turing's tensor cores accelerate fp16
#     matmuls but not bf16 ones; bf16 there would only save memory, not time.
#   - CUDA, compute capability >= 8 (Ampere+) -> bfloat16 (wider dynamic
#     range than fp16 at the same tensor-core speed, so there's no reason to
#     prefer fp16 once hardware bf16 support exists).
# ---------------------------------------------------------------------------
PRECISION_MODE = 'auto'   # override to 'fp32' / 'fp16' / 'bf16' to force a choice

def _resolve_dtype(mode):
    if mode != 'auto':
        return {'fp32': torch.float32, 'fp16': torch.float16, 'bf16': torch.bfloat16}[mode]
    if not torch.cuda.is_available():
        return torch.float32
    major, _minor = torch.cuda.get_device_capability(0)
    return torch.float16 if major < 8 else torch.bfloat16

DTYPE = _resolve_dtype(PRECISION_MODE)
_cc = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else 'n/a (CPU)'
print(f"PRECISION_MODE={PRECISION_MODE!r} -> DTYPE={DTYPE}  (compute capability {_cc})")


In [ ]:
# ---------------------------------------------------------------------------
# MODEL_CONFIGS: the single source of truth for cross-family generalization.
# Every experiment below loops over this dict instead of being hand-copied
# per model -- this *is* the generalization the original notebook's later
# "cross-model" section was reaching for, built in from the start here.
# ---------------------------------------------------------------------------
MODEL_CONFIGS = {
    'Qwen2.5-1.5B': dict(hf_id='Qwen/Qwen2.5-1.5B', family='qwen2'),
    'Gemma-2-2b':   dict(hf_id='google/gemma-2-2b',  family='gemma2'),
    'Llama-3.2-1B': dict(hf_id='meta-llama/Llama-3.2-1B', family='llama'),
}

# All three families expose the SAME module path in HF transformers:
#   model.model.layers[i].mlp.{gate_proj,up_proj,down_proj}
#   model.model.layers[i].self_attn.{q_proj,k_proj,v_proj,o_proj}
#   model.model.embed_tokens / model.lm_head
# so one adapter suffices for all three -- this is a genuine structural fact,
# not an assumption; verified by the assert block in the loader cell below.

def get_layers(model):
    return model.model.layers

def get_mlp(model, layer_idx):
    return model.model.layers[layer_idx].mlp

def get_attn(model, layer_idx):
    return model.model.layers[layer_idx].self_attn

def get_num_heads(model):
    return model.config.num_attention_heads

def get_num_kv_heads(model):
    # ROBUST HEAD-COUNT DETECTION (generalizes the original's GPT-NeoX/Pythia annex):
    # newer configs expose num_key_value_heads directly; fall back to num_attention_heads
    # (== MHA, group size 1) if the attribute is absent, as in some older configs.
    return getattr(model.config, 'num_key_value_heads', model.config.num_attention_heads)

def get_head_dim(model):
    cfg = model.config
    if hasattr(cfg, 'head_dim') and cfg.head_dim is not None:
        return cfg.head_dim
    return cfg.hidden_size // cfg.num_attention_heads

def get_d_ff(model, layer_idx):
    return get_mlp(model, layer_idx).gate_proj.out_features

## Part 0 -- Memory-safe perturbations: `ConditionModel`

Every experiment below tests a *perturbed variant* of a model (permuted MLP neurons,
permuted attention heads, a rotated hidden axis, ...). The naive way to represent
"model X with layer 4's MLP permuted" is `copy.deepcopy(model)` -- but that duplicates
**every parameter in the model**, not just the handful of tensors that actually change.
With three checkpoints already resident (needed for the cross-model comparisons in
Part 3) and ~90+ perturbation calls across the multi-seed and boundary sweeps, that
pattern would repeatedly spike GPU memory well past a single Kaggle GPU's budget.

`ConditionModel` fixes this structurally: it holds a reference to the **one shared base
model** plus the small set of *replacement* tensors for whichever layer is being
perturbed. Immediately before a forward pass it swaps the replacement tensors onto the
live model; immediately after, it swaps the originals back. No new model is ever
allocated. It's a transparent stand-in everywhere the rest of the notebook expects
"a model" (`model(ids)`, `model.eval()`, `model.config`, `get_layers(model)`, etc. all
work via attribute forwarding), so every downstream cell is unchanged in what it
*computes* -- only how the perturbed variant is *represented* changes.

The one place a real `copy.deepcopy` remains necessary is fine-tuning (Part 3, Step C),
since SGD genuinely needs to update -- and therefore genuinely needs to own -- a full
set of parameters. That cell explicitly frees its copies after use.

In [ ]:
# ---------------------------------------------------------------------------
# ConditionModel: a cheap "view" of an already-loaded model with a small set
# of tensors swapped to a stored variant, instead of a full deep copy.
# ---------------------------------------------------------------------------
class ConditionModel:
    """Represents `base_model` with the tensors in `param_refs` temporarily
    replaced by `variant_tensors` during each forward call.

    param_refs:      list of (module, attr_name) pairs, e.g. (mlp.gate_proj, 'weight')
    variant_tensors: list of replacement tensors, same order/shape as param_refs

    Only ever allocates the (small, single-layer-sized) variant tensors --
    never a second copy of the whole model. Swaps are applied right before the
    wrapped forward call and undone right after (even on exception), so the
    shared base model always holds its original weights between calls.
    """
    def __init__(self, base_model, param_refs, variant_tensors, label=""):
        assert len(param_refs) == len(variant_tensors)
        self.base_model = base_model
        self.param_refs = param_refs
        self.variant_tensors = variant_tensors
        self.label = label

    def __call__(self, *args, **kwargs):
        saved = [getattr(mod, attr).data for (mod, attr) in self.param_refs]
        for (mod, attr), variant in zip(self.param_refs, self.variant_tensors):
            getattr(mod, attr).data = variant
        try:
            return self.base_model(*args, **kwargs)
        finally:
            for (mod, attr), orig in zip(self.param_refs, saved):
                getattr(mod, attr).data = orig

    def __getattr__(self, name):
        # Forward everything else (.config, .eval(), .model, register_forward_hook
        # on submodules via get_layers(), etc.) straight to the shared base model.
        return getattr(self.base_model, name)

    def __repr__(self):
        return f"ConditionModel(label={self.label!r}, n_swapped_tensors={len(self.param_refs)})"


def mlp_param_refs(mlp):
    """(module, attr) pairs for an MLP block's weight/bias tensors."""
    refs = [(mlp.gate_proj, 'weight'), (mlp.up_proj, 'weight'), (mlp.down_proj, 'weight')]
    if mlp.gate_proj.bias is not None:
        refs.append((mlp.gate_proj, 'bias'))
    if mlp.up_proj.bias is not None:
        refs.append((mlp.up_proj, 'bias'))
    if mlp.down_proj.bias is not None:
        refs.append((mlp.down_proj, 'bias'))
    return refs


def attn_param_refs(attn):
    """(module, attr) pairs for an attention block's Q/K/V/O weight/bias tensors."""
    refs = [(attn.q_proj, 'weight'), (attn.k_proj, 'weight'), (attn.v_proj, 'weight'), (attn.o_proj, 'weight')]
    for proj in (attn.q_proj, attn.k_proj, attn.v_proj, attn.o_proj):
        if proj.bias is not None:
            refs.append((proj, 'bias'))
    return refs


def full_model_param_refs(model):
    """(module, attr) pairs spanning every layer's attention + MLP weights, plus
    embeddings and (if untied) lm_head -- used only by the global residual-stream
    rotation, which genuinely touches (almost) every tensor in the model. Still
    avoids duplicating the nn.Module graph itself, and (unlike the per-layer
    conditions) is meant to be used transiently, one at a time."""
    refs = [(model.model.embed_tokens, 'weight')]
    for layer in get_layers(model):
        refs.extend(attn_param_refs(layer.self_attn))
        refs.extend(mlp_param_refs(layer.mlp))
    if hasattr(model, 'lm_head') and not getattr(model.config, 'tie_word_embeddings', False):
        refs.append((model.lm_head, 'weight'))
    return refs


In [ ]:
# ---------------------------------------------------------------------------
# Hugging Face auth for the gated checkpoints (Gemma-2-2b, Llama-3.2-1B).
#
# SECURITY: never hardcode a raw token in a notebook cell -- notebooks get
# saved, version-controlled, and on Kaggle sometimes shared/forked publicly.
# A leaked write-scoped token can modify or delete things on your HF account.
# Use Kaggle's Secrets manager instead:
#   Notebook -> Add-ons -> Secrets -> add one named HF_TOKEN -> toggle it on
#   for this notebook. (Falls back to an HF_TOKEN environment variable for
#   non-Kaggle environments.) The value is never printed or logged below.
# -------------------------------------------------------------------------
HF_TOKEN = None
_auth_notes = []

try:
    from kaggle_secrets import UserSecretsClient
    try:
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        if HF_TOKEN:
            _auth_notes.append("found HF_TOKEN via Kaggle Secrets")
        else:
            _auth_notes.append("Kaggle Secrets available, but get_secret('HF_TOKEN') returned empty "
                                "-- the secret name doesn't match, or it exists but isn't toggled ON "
                                "for THIS notebook (Add-ons -> Secrets -> attach it here specifically)")
    except Exception as e:
        _auth_notes.append(f"Kaggle Secrets available, but reading HF_TOKEN raised: {type(e).__name__}: {e}")
except ImportError:
    _auth_notes.append("kaggle_secrets module not importable (not running on Kaggle, or add-on not enabled)")

if not HF_TOKEN:
    env_token = os.environ.get("HF_TOKEN")
    if env_token:
        HF_TOKEN = env_token
        _auth_notes.append("found HF_TOKEN via environment variable instead")
    else:
        _auth_notes.append("HF_TOKEN environment variable is also not set")

_AUTH_CHECKLIST = (
    "  1. Notebook Add-ons -> Secrets -> secret named exactly 'HF_TOKEN' -> toggled ON\n"
    "     for THIS notebook (adding it globally is not enough).\n"
    "  2. On huggingface.co, the account that issued the token must have clicked\n"
    "     'Agree and access repository' on both google/gemma-2-2b and\n"
    "     meta-llama/Llama-3.2-1B -- a valid token with an unaccepted license\n"
    "     produces the same 401 error as a missing token."
)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face auth: " + _auth_notes[0])
else:
    print("Hugging Face auth: NO TOKEN FOUND. Gemma-2-2b and Llama-3.2-1B are gated and")
    print("will fail to load with a 401/GatedRepoError until this is fixed. Diagnostics:")
    for note in _auth_notes:
        print(f"  - {note}")
    print("Checklist:")
    print(_AUTH_CHECKLIST)

print("=" * 78)
print("LOADING MODELS (one at a time, low_cpu_mem_usage, memory-checked)")
print("=" * 78)
print_gpu_mem("before any load")

tokenizers, models = {}, {}
for name, cfg in MODEL_CONFIGS.items():
    print(f"\nLoading {name} ({cfg['hf_id']}) ...")
    try:
        tok = AutoTokenizer.from_pretrained(cfg['hf_id'], token=HF_TOKEN)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        m = AutoModelForCausalLM.from_pretrained(
            cfg['hf_id'],
            dtype=DTYPE,
            low_cpu_mem_usage=True,
            token=HF_TOKEN,
        ).to(DEVICE)
    except Exception as e:
        # Surface a short, actionable message instead of the raw ~150-line
        # httpx/huggingface_hub traceback -- the underlying exception (401 /
        # GatedRepoError) is almost always missing-or-unaccepted-license, not
        # a code bug, so a wall of stack frames just buries the fix.
        is_auth_error = ('401' in str(e)) or ('Gated' in type(e).__name__) or ('gated repo' in str(e).lower())
        if is_auth_error:
            raise RuntimeError(
                f"\nCould not load '{cfg['hf_id']}' ({name}): access denied (401 / gated repo).\n"
                f"This means the HF_TOKEN is missing/invalid, or its account hasn't accepted\n"
                f"the model's license yet. Fix, then re-run this cell (no kernel restart needed):\n"
                f"{_AUTH_CHECKLIST}"
            ) from None
        raise
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)  # inference-only; saves autograd bookkeeping memory
    tokenizers[name] = tok
    models[name] = m
    print(f"  layers={len(get_layers(m))}  hidden={m.config.hidden_size}  "
          f"heads={get_num_heads(m)}  kv_heads={get_num_kv_heads(m)}  "
          f"head_dim={get_head_dim(m)}  d_ff={get_d_ff(m, 0)}  dtype={next(m.parameters()).dtype}")
    # structural assumption check for the adapters above
    assert hasattr(m, 'model') and hasattr(m.model, 'layers'), f"{name}: unexpected top-level structure"
    assert hasattr(get_mlp(m, 0), 'gate_proj') and hasattr(get_mlp(m, 0), 'up_proj') and hasattr(get_mlp(m, 0), 'down_proj')
    assert hasattr(get_attn(m, 0), 'q_proj') and hasattr(get_attn(m, 0), 'k_proj') and hasattr(get_attn(m, 0), 'v_proj') and hasattr(get_attn(m, 0), 'o_proj')
    free_memory()
    print_gpu_mem(f"after loading {name}")

print("\nAll three families confirmed to share the same module layout.")
print_gpu_mem("after all loads")
print("\n(If this is uncomfortably close to the GPU's total, set PRECISION_MODE = 'bf16'")
print(" in the earlier cell and re-run from the top, or restart the kernel first to")
print(" release any memory held from a previous run.)")


### Smoke test: does the `ConditionModel` swap actually reach the forward pass?

Runs a large, unmissable perturbation (zeroing an entire MLP projection) against each
just-loaded model and checks the output changes by a lot. If this fails, nothing
downstream that relies on `ConditionModel` can be trusted -- fix this first.

In [ ]:
print("=" * 78)
print("SMOKE TEST: ConditionModel swap reaches the real forward pass")
print("=" * 78)

def _smoke_test_condition_model(model, tokenizer, device):
    mlp = get_mlp(model, 0)
    ids = tokenizer("The quick brown fox jumps over the lazy dog.", return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        base_logits = model(ids).logits.clone()
    zeroed = torch.zeros_like(mlp.down_proj.weight.data)
    cm = ConditionModel(model, [(mlp.down_proj, 'weight')], [zeroed], label='smoke_test_zero_down_proj')
    with torch.no_grad():
        perturbed_logits = cm(ids).logits
        restored_logits = model(ids).logits  # weights should already be back to normal here
    diff_perturbed = (base_logits - perturbed_logits).abs().max().item()
    diff_restored = (base_logits - restored_logits).abs().max().item()
    ok = diff_perturbed > 1e-3 and diff_restored < 1e-4
    return ok, diff_perturbed, diff_restored

all_ok = True
for name in MODEL_CONFIGS:
    ok, diff_p, diff_r = _smoke_test_condition_model(models[name], tokenizers[name], DEVICE)
    status = "PASS" if ok else "FAIL"
    print(f"  {name:>16s}: {status}  (zeroed-layer diff={diff_p:.3e}, post-restore diff={diff_r:.3e})")
    all_ok = all_ok and ok

if all_ok:
    print("\nAll models: swap correctly reaches the forward pass and restores cleanly.")
    print("Any near-1.0 CKA values seen later reflect CKA's known insensitivity to small")
    print("perturbations relative to a few dominant activation channels -- not a broken swap.")
else:
    print("\nAt least one model FAILED this smoke test -- the ConditionModel mechanism is")
    print("not affecting that model's forward pass. Do not trust downstream results for it")
    print("until this is fixed (check transformers version / attn_implementation / any")
    print("custom model code that might not read weights via standard nn.Linear .weight).")
    raise RuntimeError("ConditionModel smoke test failed -- see printout above.")


In [ ]:
# Same fixed IID eval corpus as the original (swap for a larger held-out set for a
# publication-grade run; kept small here so the notebook is fast to read through).
EVAL_CORPUS = [
    "The quick brown fox jumps over the lazy dog.",
    "Natural language processing is a subfield of artificial intelligence.",
    "Machine learning models learn patterns from large amounts of data.",
    "The transformer architecture relies on self-attention mechanisms.",
    "Scientists discovered a new species of frog in the rainforest.",
    "The stock market experienced significant volatility this week.",
    "Cooking pasta requires boiling water and a pinch of salt.",
    "The history of Rome spans over a thousand years.",
]

OOD_CORPUS = {
    'code': [
        "def factorial(n):\n    if n == 0:\n        return 1\n    return n * factorial(n - 1)",
        "for i in range(10):\n    print(i ** 2)",
        "{\"name\": \"example\", \"values\": [1, 2, 3], \"active\": true}",
    ],
    'non_english': [
        "La vida es un viaje lleno de sorpresas inesperadas.",
        "Der schnelle braune Fuchs springt uber den faulen Hund.",
        "Le soleil se leve a l'est chaque matin sans exception.",
    ],
    'corrupted': [
        "the dog quickly the over jumps fox brown",
        "xk3f9 random tokens zzq without any coherent meaning qwerty",
        "was is the of a to in on at by for with",
    ],
}

CAUSAL_TRACE_PAIRS = [
    ("The capital of France is", " Paris", "The capital of Germany is", " Berlin"),
    ("The opposite of hot is", " cold", "The opposite of up is", " down"),
    ("A dog says", " woof", "A cat says", " meow"),
]

FEW_SHOT_ANTONYM = [
    ("hot -> cold\nbig -> small\nfast -> ", " slow"),
    ("up -> down\nleft -> right\nwet -> ", " dry"),
    ("happy -> sad\nlight -> dark\nyoung -> ", " old"),
]

## Part 1 -- Baseline behavioral battery

**Motivating literature.** Same rationale as the original: a clean baseline before any
manipulation is standard interpretability practice, and accuracy/perplexity/ECE together
form this project's "baseline identity" object that every later transformation is
compared against.

**What's identical vs. adapted.** The three metric functions below (`next_token_accuracy`,
`perplexity`, `expected_calibration_error`) are architecture-agnostic -- they only touch
`model(...).logits`, which every `AutoModelForCausalLM` exposes identically regardless of
family. Nothing needed to change here; this is a genuinely exact replication of the
original cell, just called on three checkpoints instead of one.

In [ ]:
def next_token_accuracy(model, tokenizer, texts, device='cpu'):
    accs = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            logits = model(ids).logits
            preds = logits[0, :-1, :].argmax(dim=-1)
            targets = ids[0, 1:]
            accs.append((preds == targets).float().mean().item())
    return float(np.mean(accs))


def perplexity(model, tokenizer, texts, device='cpu'):
    losses = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            out = model(ids, labels=ids)
            losses.append(out.loss.item())
    return float(np.exp(np.mean(losses)))


def expected_calibration_error(model, tokenizer, texts, device='cpu', n_bins=10):
    confidences, corrects = [], []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            logits = model(ids).logits[0, :-1, :]
            targets = ids[0, 1:]
            probs = torch.softmax(logits, dim=-1)
            conf, pred = probs.max(dim=-1)
            confidences.extend(conf.tolist())
            corrects.extend((pred == targets).tolist())
    confidences = np.array(confidences)
    corrects = np.array(corrects).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(confidences)) * abs(corrects[mask].mean() - confidences[mask].mean())
    return float(ece)


def run_battery(model, tokenizer, texts, device='cpu', label='model', verbose=True):
    acc = next_token_accuracy(model, tokenizer, texts, device)
    ppl = perplexity(model, tokenizer, texts, device)
    ece = expected_calibration_error(model, tokenizer, texts, device)
    if verbose:
        print(f"{label:>28s} | accuracy={acc:.4f}  perplexity={ppl:9.3f}  ECE={ece:.4f}")
    return {'label': label, 'accuracy': acc, 'perplexity': ppl, 'ece': ece}


print("=" * 78)
print("STEP 1: BASELINE, ALL THREE FAMILIES")
print("=" * 78)
baseline_results = {}
for name in MODEL_CONFIGS:
    baseline_results[name] = run_battery(models[name], tokenizers[name], EVAL_CORPUS, DEVICE, label=name)

## Part 1 -- Aligned / unaligned MLP permutation (SwiGLU / GeGLU generalization)

**Motivating literature.** Same as the original: Hecht-Nielsen (1990); Entezari et al.
(2022); Ainsworth et al. (2023) -- the "weight matching" operation $W \to WP$.

**What's genuinely different here (not just renamed).** GPT-2's MLP is
`down = c_proj(gelu(c_fc(x)))` -- one hidden projection, so one permutation matrix on one
axis. Llama/Qwen/Gemma's MLP is
`down = down_proj(act(gate_proj(x)) * up_proj(x))` -- **two** projections whose outputs
are multiplied elementwise before hitting `down_proj`. For a permutation $P$ on the
$d_{ff}$ axis to be an exact symmetry here, it must be applied to **both** `gate_proj`'s
and `up_proj`'s output rows (so the elementwise product is permuted consistently), and
`down_proj`'s input columns must be compensated by $P^\top$. Applying $P$ to only one of
{gate_proj, up_proj} is a new, architecture-specific failure mode that has no GPT-2
analogue (it isn't just "the unaligned control" -- it's a *third* condition: aligned,
gate-only, and fully-unaligned all break the correspondence in different ways). We test
the fully-unaligned case (mirroring the original's design) and separately check the
gate-only case in a note below.

**Method identical or different?** Identical algebraic principle (compensated linear
permutation of a hidden axis); the two-projection SwiGLU/GeGLU structure is the
architecture-specific extension.

In [ ]:
def make_aligned_mlp_permutation(model, layer_idx, seed=0, device='cpu'):
    '''Returns a ConditionModel (not a full model copy) representing `model` with
    the d_ff hidden axis of one MLP block permuted: the SAME permutation P applied
    to gate_proj's and up_proj's output rows (nn.Linear layout: weight is (out, in),
    so permuting outputs = permuting weight ROWS, y = x @ W^T + b), and down_proj's
    input columns compensated by P^T. Only this one layer's tensors are ever
    duplicated -- see the ConditionModel cell in Part 0 for why.'''
    mlp = get_mlp(model, layer_idx)
    d_ff = mlp.gate_proj.out_features
    torch.manual_seed(seed)
    perm = torch.randperm(d_ff)
    param_refs = [(mlp.gate_proj, 'weight'), (mlp.up_proj, 'weight'), (mlp.down_proj, 'weight')]
    variants = [
        mlp.gate_proj.weight.data[perm, :].clone(),
        mlp.up_proj.weight.data[perm, :].clone(),
        mlp.down_proj.weight.data[:, perm].clone(),
    ]
    if mlp.gate_proj.bias is not None:
        param_refs.append((mlp.gate_proj, 'bias')); variants.append(mlp.gate_proj.bias.data[perm].clone())
    if mlp.up_proj.bias is not None:
        param_refs.append((mlp.up_proj, 'bias')); variants.append(mlp.up_proj.bias.data[perm].clone())
    cm = ConditionModel(model, param_refs, variants, label=f'aligned_mlp_L{layer_idx}')
    return cm, perm


def make_unaligned_mlp_permutation(model, layer_idx, seed=0, device='cpu'):
    '''Same permutation applied to gate_proj/up_proj, but down_proj is left
    UNCOMPENSATED -- the broken-symmetry negative control. ConditionModel-based,
    same memory profile as the aligned case above.'''
    mlp = get_mlp(model, layer_idx)
    d_ff = mlp.gate_proj.out_features
    torch.manual_seed(seed)
    perm = torch.randperm(d_ff)
    param_refs = [(mlp.gate_proj, 'weight'), (mlp.up_proj, 'weight')]
    variants = [
        mlp.gate_proj.weight.data[perm, :].clone(),
        mlp.up_proj.weight.data[perm, :].clone(),
    ]
    if mlp.gate_proj.bias is not None:
        param_refs.append((mlp.gate_proj, 'bias')); variants.append(mlp.gate_proj.bias.data[perm].clone())
    if mlp.up_proj.bias is not None:
        param_refs.append((mlp.up_proj, 'bias')); variants.append(mlp.up_proj.bias.data[perm].clone())
    # down_proj intentionally left untouched -- not included in param_refs at all
    cm = ConditionModel(model, param_refs, variants, label=f'unaligned_mlp_L{layer_idx}')
    return cm, perm


def make_gateonly_mlp_permutation(model, layer_idx, seed=0, device='cpu'):
    '''Architecture-specific third condition with no GPT-2 analogue: permute ONLY
    gate_proj's output, leaving up_proj and down_proj untouched. Tests whether the
    two SwiGLU/GeGLU branches are independently load-bearing.'''
    mlp = get_mlp(model, layer_idx)
    d_ff = mlp.gate_proj.out_features
    torch.manual_seed(seed)
    perm = torch.randperm(d_ff)
    param_refs = [(mlp.gate_proj, 'weight')]
    variants = [mlp.gate_proj.weight.data[perm, :].clone()]
    if mlp.gate_proj.bias is not None:
        param_refs.append((mlp.gate_proj, 'bias')); variants.append(mlp.gate_proj.bias.data[perm].clone())
    cm = ConditionModel(model, param_refs, variants, label=f'gateonly_mlp_L{layer_idx}')
    return cm, perm


TARGET_LAYER = 4  # an interior layer; change and re-run to spot-check other depths

print("=" * 78)
print("STEP 2-3: ALIGNED / UNALIGNED / GATE-ONLY MLP PERMUTATION, LAYER", TARGET_LAYER)
print("=" * 78)

mlp_perm_models = {}
for name in MODEL_CONFIGS:
    m = models[name]
    aligned_m, perm = make_aligned_mlp_permutation(m, TARGET_LAYER, seed=0, device=DEVICE)
    unaligned_m, _ = make_unaligned_mlp_permutation(m, TARGET_LAYER, seed=0, device=DEVICE)
    gateonly_m, _ = make_gateonly_mlp_permutation(m, TARGET_LAYER, seed=0, device=DEVICE)
    mlp_perm_models[name] = dict(aligned=aligned_m, unaligned=unaligned_m, gateonly=gateonly_m)

    tok = tokenizers[name]
    sample_ids = tok(EVAL_CORPUS[0], return_tensors='pt').input_ids.to(DEVICE)
    with torch.no_grad():
        base_logits = m(sample_ids).logits
        a_diff = (base_logits - aligned_m(sample_ids).logits).abs().max().item()
        u_diff = (base_logits - unaligned_m(sample_ids).logits).abs().max().item()
        g_diff = (base_logits - gateonly_m(sample_ids).logits).abs().max().item()
    print(f"\n{name} (layer {TARGET_LAYER}, d_ff={get_d_ff(m, TARGET_LAYER)}):")
    print(f"  aligned   max|logit diff| = {a_diff:.2e}  (expect ~0 at this dtype's noise floor)")
    print(f"  unaligned max|logit diff| = {u_diff:.3f}  (expect large)")
    print(f"  gate-only max|logit diff| = {g_diff:.3f}  (expect large -- new architecture-specific break)")

print_gpu_mem("after MLP condition setup (should be ~unchanged from model-load, by design)")


In [ ]:
print("=" * 78)
print("STEP 4: FULL BATTERY, ALL THREE FAMILIES x {aligned, unaligned, gate-only}")
print("=" * 78)

mlp_perm_battery = {}
for name in MODEL_CONFIGS:
    tok = tokenizers[name]
    print(f"\n-- {name} --")
    mlp_perm_battery[name] = {'baseline': baseline_results[name]}
    for cond in ['aligned', 'unaligned', 'gateonly']:
        mlp_perm_battery[name][cond] = run_battery(
            mlp_perm_models[name][cond], tok, EVAL_CORPUS, DEVICE, label=cond)

print("\nExpected: aligned diffs ~0 across all three models and all three metrics;")
print("unaligned and gate-only diffs large, confirming the SwiGLU/GeGLU two-branch")
print("dependency has no partial-credit regime.")

## Part 1 -- Representation similarity: linear CKA per layer

**Motivating literature.** **Kornblith, Norouzi, Lee & Hinton (2019)**, linear CKA,
used here (as in the original) as a *falsifiable exact-match prediction* (CKA should
equal 1.0 for the aligned condition) rather than a descriptive similarity score.

In [ ]:
def linear_cka(X, Y):
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    num = ((X.T @ Y) ** 2).sum()
    den = np.sqrt(((X.T @ X) ** 2).sum() * ((Y.T @ Y) ** 2).sum())
    return float(num / den) if den > 0 else 0.0


def collect_hidden_states(model, tokenizer, texts, layer, device='cpu'):
    rows = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            out = model(ids, output_hidden_states=True)
            rows.append(out.hidden_states[layer][0].float().cpu().numpy())
    return np.concatenate(rows, axis=0)


print("=" * 78)
print("STEP 5: LINEAR CKA PER LAYER, BASELINE vs. ALIGNED/UNALIGNED, ALL FAMILIES")
print("=" * 78)
print("NOTE: CKA in [0,1] is bounded and, in real trained LLMs, is often dominated by a")
print("handful of very-high-magnitude ('outlier'/'massive activation') channels -- a real,")
print("nonzero perturbation elsewhere in the representation can still round to 1.0000 at")
print("4 decimal places, especially under bf16/fp16. So each row below also prints the raw")
print("max|hidden-state diff| (unrounded, in scientific notation) -- if THAT is exactly 0.0,")
print("the perturbation genuinely isn't reaching the forward pass (a real bug); if it's")
print("nonzero but CKA still reads ~1.0000, that's CKA-insensitivity, not a broken swap.")

cka_records = {}
for name in MODEL_CONFIGS:
    m, tok = models[name], tokenizers[name]
    n_layers_plus_embed = len(get_layers(m)) + 1
    cka_records[name] = {'aligned': [], 'unaligned': []}
    print(f"\n-- {name} --")
    for layer in range(n_layers_plus_embed):
        h_base = collect_hidden_states(m, tok, EVAL_CORPUS, layer, DEVICE)
        h_al = collect_hidden_states(mlp_perm_models[name]['aligned'], tok, EVAL_CORPUS, layer, DEVICE)
        h_un = collect_hidden_states(mlp_perm_models[name]['unaligned'], tok, EVAL_CORPUS, layer, DEVICE)
        cka_a, cka_u = linear_cka(h_base, h_al), linear_cka(h_base, h_un)
        raw_a = float(np.abs(h_base - h_al).max())
        raw_u = float(np.abs(h_base - h_un).max())
        cka_records[name]['aligned'].append(cka_a)
        cka_records[name]['unaligned'].append(cka_u)
        print(f"  layer {layer:2d}: CKA(aligned)={cka_a:.6f} (raw diff={raw_a:.2e})   "
              f"CKA(unaligned)={cka_u:.6f} (raw diff={raw_u:.2e})")
        if layer >= TARGET_LAYER + 1 and raw_u == 0.0:
            print(f"    !! WARNING: unaligned raw diff is exactly 0.0 past the perturbed layer --")
            print(f"       the perturbation is not reaching the forward pass. See the smoke-test")
            print(f"       cell right after model loading; re-run it and check it reports PASS.")

fig, axes = plt.subplots(1, len(MODEL_CONFIGS), figsize=(5.5 * len(MODEL_CONFIGS), 4.2), sharey=True)
for ax, name in zip(axes, MODEL_CONFIGS):
    rec = cka_records[name]
    ax.plot(rec['aligned'], marker='o', ms=3, color=COLORS['aligned'], label='aligned')
    ax.plot(rec['unaligned'], marker='o', ms=3, color=COLORS['unaligned'], label='unaligned')
    ax.axhline(1.0, color='gray', ls='--', lw=1)
    ax.set_title(name)
    ax.set_xlabel('Layer (0 = embeddings)')
axes[0].set_ylabel('Linear CKA vs. baseline')
axes[0].legend()
plt.tight_layout()
plt.savefig('cka_per_layer_all_models.png', dpi=150, bbox_inches='tight')
plt.show()


## Part 1 -- Causal tracing (ROME-style)

**Motivating literature.** **Meng, Bau, Andonian & Belinkov (2022)** ("ROME");
**Vig et al. (2020)** causal mediation analysis. Same clean/corrupted/patch protocol as
the original, re-targeted at `model.model.layers[i]` (the HF module path shared by all
three families) instead of `model.transformer.h[i]`.

In [ ]:
def logit_diff(logits, correct_id, wrong_id):
    return (logits[0, -1, correct_id] - logits[0, -1, wrong_id]).item()


def run_causal_tracing(model, tokenizer, pairs, device='cpu'):
    n_layers = len(get_layers(model))
    restoration_by_layer = np.zeros(n_layers)
    used_pairs = 0

    for clean_prompt, correct_ans, corrupted_prompt, wrong_ans in pairs:
        correct_id = tokenizer.encode(correct_ans, add_special_tokens=False)[0]
        wrong_id = tokenizer.encode(wrong_ans, add_special_tokens=False)[0]

        clean_ids = tokenizer(clean_prompt, return_tensors='pt').input_ids.to(device)
        corrupt_ids = tokenizer(corrupted_prompt, return_tensors='pt').input_ids.to(device)
        if clean_ids.shape[1] != corrupt_ids.shape[1]:
            continue
        used_pairs += 1

        with torch.no_grad():
            clean_out = model(clean_ids, output_hidden_states=True)
            clean_cache = clean_out.hidden_states
            clean_logits = clean_out.logits
            corrupt_logits = model(corrupt_ids).logits

        clean_diff = logit_diff(clean_logits, correct_id, wrong_id)
        corrupt_diff = logit_diff(corrupt_logits, correct_id, wrong_id)
        subject_pos = clean_ids.shape[1] - 2

        for layer in range(n_layers):
            def hook(module, inp, out, layer=layer):
                if isinstance(out, tuple):
                    out[0][:, subject_pos, :] = clean_cache[layer + 1][:, subject_pos, :]
                    return out
                out[:, subject_pos, :] = clean_cache[layer + 1][:, subject_pos, :]
                return out

            h = get_layers(model)[layer].register_forward_hook(hook)
            with torch.no_grad():
                patched_logits = model(corrupt_ids).logits
            h.remove()

            patched_diff = logit_diff(patched_logits, correct_id, wrong_id)
            denom = (clean_diff - corrupt_diff) if (clean_diff - corrupt_diff) != 0 else 1e-8
            restoration_by_layer[layer] += (patched_diff - corrupt_diff) / denom

    if used_pairs > 0:
        restoration_by_layer /= used_pairs
    return restoration_by_layer, used_pairs


print("=" * 78)
print("STEP 6: CAUSAL TRACING -- DOES THE UNALIGNED (BROKEN) MODEL STILL LOCATE THE FACT")
print("=" * 78)

causal_trace_records = {}
for name in MODEL_CONFIGS:
    tok = tokenizers[name]
    unaligned_m = mlp_perm_models[name]['unaligned']
    restoration, used = run_causal_tracing(unaligned_m, tok, CAUSAL_TRACE_PAIRS, DEVICE)
    causal_trace_records[name] = restoration
    print(f"\n-- {name} ({used}/{len(CAUSAL_TRACE_PAIRS)} length-matched pairs used) --")
    for layer, val in enumerate(restoration):
        print(f"  layer {layer:2d}: mean restoration = {val:.3f}")

## Part 1 -- Out-of-distribution stress test

In [ ]:
def ood_perplexity_report(model, tokenizer, ood_corpus, device='cpu', label='model'):
    results = {}
    for domain, texts in ood_corpus.items():
        results[domain] = perplexity(model, tokenizer, texts, device)
    return results


print("=" * 78)
print("STEP 7: OOD STRESS TEST, ALL THREE FAMILIES")
print("=" * 78)

ood_records = {}
for name in MODEL_CONFIGS:
    tok = tokenizers[name]
    print(f"\n-- {name} --")
    ood_records[name] = {}
    for cond, m in [('baseline', models[name]), ('aligned', mlp_perm_models[name]['aligned']),
                     ('unaligned', mlp_perm_models[name]['unaligned'])]:
        r = ood_perplexity_report(m, tok, OOD_CORPUS, DEVICE, cond)
        ood_records[name][cond] = r
        print(f"  {cond:>10s}: " + "  ".join(f"{d}={v:8.2f}" for d, v in r.items()))

## Part 1 -- Multi-seed sweep with confidence intervals

In [ ]:
N_SEEDS = 15
CONFIDENCE = 0.95


def mlp_seed_sweep(model, tokenizer, texts, layer_idx, n_seeds, device='cpu'):
    aligned_diffs, unaligned_diffs = [], []
    baseline_acc = next_token_accuracy(model, tokenizer, texts, device)
    for seed in range(n_seeds):
        a, _ = make_aligned_mlp_permutation(model, layer_idx, seed=seed, device=device)
        u, _ = make_unaligned_mlp_permutation(model, layer_idx, seed=seed, device=device)
        aligned_diffs.append(next_token_accuracy(a, tokenizer, texts, device) - baseline_acc)
        unaligned_diffs.append(next_token_accuracy(u, tokenizer, texts, device) - baseline_acc)
    return np.array(aligned_diffs), np.array(unaligned_diffs)


def mean_ci(values, confidence=0.95):
    mean = values.mean()
    sem = scipy_stats.sem(values) if len(values) > 1 else 0.0
    ci = scipy_stats.t.interval(confidence, len(values) - 1, loc=mean, scale=sem) if sem > 0 else (mean, mean)
    return float(mean), (float(ci[0]), float(ci[1]))


print("=" * 78)
print(f"STEP 8: MULTI-SEED SWEEP (N={N_SEEDS}, {int(CONFIDENCE*100)}% CI), ALL FAMILIES")
print("=" * 78)

seed_sweep_records = {}
for name in MODEL_CONFIGS:
    m, tok = models[name], tokenizers[name]
    a_diffs, u_diffs = mlp_seed_sweep(m, tok, EVAL_CORPUS, TARGET_LAYER, N_SEEDS, DEVICE)
    a_mean, a_ci = mean_ci(a_diffs, CONFIDENCE)
    u_mean, u_ci = mean_ci(u_diffs, CONFIDENCE)
    seed_sweep_records[name] = dict(a_diffs=a_diffs, u_diffs=u_diffs, a_mean=a_mean, a_ci=a_ci, u_mean=u_mean, u_ci=u_ci)
    print(f"\n-- {name} --")
    print(f"  aligned   accuracy diff: mean={a_mean:+.5f}  CI=[{a_ci[0]:+.5f}, {a_ci[1]:+.5f}]")
    print(f"  unaligned accuracy diff: mean={u_mean:+.5f}  CI=[{u_ci[0]:+.5f}, {u_ci[1]:+.5f}]")

## Part 2 -- Attention-head permutation, generalized for GQA

**Motivating literature.** **Olsson, Elhage, Nanda et al. (2022)**, induction heads;
Hecht-Nielsen-style permutation algebra extended to the attention block.

**The genuinely new part.** GPT-2 has multi-head attention: num query heads == num KV
heads, so permuting "heads" means the same thing for Q, K, and V. Llama/Qwen/Gemma use
**grouped-query attention**: `num_attention_heads` query heads share
`num_key_value_heads` K/V heads in groups of size `group_size =
num_attention_heads // num_key_value_heads`. A query-head permutation is only a valid
symmetry if it **permutes whole groups together** -- i.e. the permutation must act on
KV-head indices, and query heads within a group move with their parent KV head, not
independently. Permuting individual query heads *within* different groups (breaking
group structure) is a new, GQA-specific failure mode this cell tests explicitly as a
second negative control (`unaligned_within_group`), alongside the standard
"no o_proj compensation" control from the original notebook.

In [ ]:
def build_block_permutation(n_blocks, block_dim, perm, device='cpu', dtype=torch.float32):
    '''Block-permutation matrix over `n_blocks` blocks of size `block_dim` each.
    `dtype` MUST match the tensors it will be matrix-multiplied against (e.g. the
    model's bf16/fp16 weights) -- torch.matmul requires both operands to share a
    dtype, and this defaulted to float32 while the model runs in bf16/fp16, which
    is exactly what produced the "expected mat1 and mat2 to have the same dtype"
    RuntimeError below.'''
    total = n_blocks * block_dim
    P = torch.zeros(total, total, device=device, dtype=dtype)
    for new_pos, old_block in enumerate(perm):
        rows = torch.arange(old_block * block_dim, (old_block + 1) * block_dim)
        cols = torch.arange(new_pos * block_dim, (new_pos + 1) * block_dim)
        P[rows, cols] = 1.0
    return P


def permute_attention_heads_gqa(model, layer_idx=0, mode='aligned', seed=0, device='cpu'):
    '''mode in {'aligned', 'unaligned_no_compensation', 'unaligned_within_group'}.
    Returns a ConditionModel (per the memory-safety note in Part 0) -- only this
    layer's Q/K/V/O tensors are ever duplicated, never the whole model.

    aligned: permute KV-head groups (group = num_heads // num_kv_heads adjacent query
      heads sharing one KV head); Q heads move with their parent group; K/V heads permuted
      the same way; o_proj input compensated by P^T. Exact symmetry.

    unaligned_no_compensation: same Q/K/V permutation as aligned, but o_proj left
      uncompensated (direct GQA analogue of the original notebook's negative control).

    unaligned_within_group: permutes individual QUERY heads across DIFFERENT groups
      (i.e. reassigns a query head to a different KV parent) -- has no GPT-2 analogue,
      since GPT-2's group size is 1. Always breaks the symmetry even with o_proj
      "compensated" for the group-level permutation, because query heads no longer
      attend using the K/V they were trained against.
    '''
    attn = get_attn(model, layer_idx)
    n_heads, n_kv = get_num_heads(model), get_num_kv_heads(model)
    head_dim = get_head_dim(model)
    group_size = n_heads // n_kv

    attn_dtype = attn.q_proj.weight.dtype
    torch.manual_seed(seed)
    group_perm = torch.randperm(n_kv).tolist()   # permutation over KV-head groups
    P_kv = build_block_permutation(n_kv, head_dim, group_perm, device=device, dtype=attn_dtype)
    q_head_perm = []
    for g in group_perm:
        q_head_perm.extend(range(g * group_size, (g + 1) * group_size))
    P_q = build_block_permutation(n_heads, head_dim, q_head_perm, device=device, dtype=attn_dtype)

    param_refs, variants = [], []

    def add(module, attr, new_tensor):
        param_refs.append((module, attr)); variants.append(new_tensor)

    if mode in ('aligned', 'unaligned_no_compensation'):
        add(attn.q_proj, 'weight', P_q.T @ attn.q_proj.weight.data)
        add(attn.k_proj, 'weight', P_kv.T @ attn.k_proj.weight.data)
        add(attn.v_proj, 'weight', P_kv.T @ attn.v_proj.weight.data)
        if attn.q_proj.bias is not None:
            add(attn.q_proj, 'bias', P_q.T @ attn.q_proj.bias.data)
        if attn.k_proj.bias is not None:
            add(attn.k_proj, 'bias', P_kv.T @ attn.k_proj.bias.data)
        if attn.v_proj.bias is not None:
            add(attn.v_proj, 'bias', P_kv.T @ attn.v_proj.bias.data)
        if mode == 'aligned':
            add(attn.o_proj, 'weight', attn.o_proj.weight.data @ P_q)
        # else: o_proj left uncompensated on purpose (not added to param_refs)

    elif mode == 'unaligned_within_group':
        within_group_perm = torch.randperm(n_heads).tolist()  # ignores group structure
        P_wg = build_block_permutation(n_heads, head_dim, within_group_perm, device=device, dtype=attn_dtype)
        add(attn.q_proj, 'weight', P_wg.T @ attn.q_proj.weight.data)
        if attn.q_proj.bias is not None:
            add(attn.q_proj, 'bias', P_wg.T @ attn.q_proj.bias.data)
        add(attn.o_proj, 'weight', attn.o_proj.weight.data @ P_wg)
        # K/V left untouched -- Q heads now paired with the wrong KV parent

    return ConditionModel(model, param_refs, variants, label=f'{mode}_heads_L{layer_idx}')


print("=" * 78)
print("SECTION 2: ATTENTION-HEAD PERMUTATION (GQA-AWARE), ALL THREE FAMILIES")
print("=" * 78)

head_perm_models = {}
for name in MODEL_CONFIGS:
    m, tok = models[name], tokenizers[name]
    n_heads, n_kv = get_num_heads(m), get_num_kv_heads(m)
    print(f"\n-- {name}: {n_heads} query heads, {n_kv} KV heads, group_size={n_heads // n_kv} --")

    aligned_m = permute_attention_heads_gqa(m, TARGET_LAYER, mode='aligned', seed=0, device=DEVICE)
    unaligned_nc_m = permute_attention_heads_gqa(m, TARGET_LAYER, mode='unaligned_no_compensation', seed=0, device=DEVICE)
    unaligned_wg_m = permute_attention_heads_gqa(m, TARGET_LAYER, mode='unaligned_within_group', seed=0, device=DEVICE)
    head_perm_models[name] = dict(aligned=aligned_m, unaligned_nc=unaligned_nc_m, unaligned_wg=unaligned_wg_m)

    sample_ids = tok(EVAL_CORPUS[0], return_tensors='pt').input_ids.to(DEVICE)
    with torch.no_grad():
        base_logits = m(sample_ids).logits
        for cond_name, cond_m in head_perm_models[name].items():
            diff = (base_logits - cond_m(sample_ids).logits).abs().max().item()
            print(f"  {cond_name:>14s}: max|logit diff| = {diff:.2e}")

    for cond_name, cond_m in [('baseline', m)] + list(head_perm_models[name].items()):
        run_battery(cond_m, tok, EVAL_CORPUS, DEVICE, label=f"{name}/{cond_name}")

print_gpu_mem("after head-permutation condition setup")


## Part 2 -- Alternative perturbation controls: rotation, noise, deletion

**Motivating literature.** Same framing as the original -- these are *negative controls
against permutation specifically* (Entezari et al. 2022 and Ainsworth et al. 2023 both
require permutation, not arbitrary linear maps). Re-derived for `nn.Linear`'s
$y = xW^\top+b$ layout (rotate/permute weight **rows** for outputs, columns for inputs
-- the transpose of the GPT-2/`Conv1D` case) and RMSNorm-normalized residual streams
(global rotation here conjugates the residual stream through RMSNorm, which -- unlike
GPT-2's LayerNorm -- has no learned bias to worry about, only a learned per-channel
scale that does *not* commute with an arbitrary rotation; this is flagged explicitly
because it means "global rotation" is *not* an exact symmetry for RMSNorm-normalized
models the way it would be for a mean/variance-only normalization, and the numeric
check below is expected to show a nonzero -- not necessarily catastrophic -- deviation).

In [ ]:
def make_local_orthogonal_mlp(model, layer_idx, seed=0, device='cpu'):
    '''ConditionModel version: rotates one MLP block's d_ff axis by a random
    orthogonal Q. Only this layer's tensors are duplicated.'''
    mlp = get_mlp(model, layer_idx)
    d_ff = mlp.gate_proj.out_features
    # Q is built and QR-decomposed on CPU (see cpu_orthogonal) -- this GPU
    # never has to hold a full fp32 d_ff x d_ff buffer, only the final
    # model-dtype Q used in the two small matmuls below. (Old approach built
    # + QR'd A directly on `device`; cuSOLVER's QR only supports fp32/fp64
    # anyway, so that bought nothing but VRAM pressure.)
    Q = cpu_orthogonal(d_ff, seed=seed).to(device=device, dtype=mlp.gate_proj.weight.dtype)
    param_refs = [(mlp.gate_proj, 'weight'), (mlp.up_proj, 'weight'), (mlp.down_proj, 'weight')]
    variants = [
        Q.T @ mlp.gate_proj.weight.data,
        Q.T @ mlp.up_proj.weight.data,
        mlp.down_proj.weight.data @ Q,
    ]
    return ConditionModel(model, param_refs, variants, label=f'local_rotation_L{layer_idx}')


def make_global_rotation(model, seed=0, device='cpu'):
    '''Rotates the whole residual stream by a single orthogonal Q. NOTE: because
    RMSNorm's per-channel learned scale does not commute with a generic rotation
    (unlike a hypothetical scale-free normalization), this is expected to be only
    approximately, not exactly, identity-preserving -- unlike the aligned-permutation
    cases above. The numeric check quantifies exactly how far off it is.

    Unlike the single-layer perturbations above, this genuinely touches (almost)
    every tensor in the model, so the ConditionModel it returns is not "cheap" the
    way the others are -- but it's still only ONE transient set of full-size variant
    tensors, created and freed one at a time (see the loop below), rather than the
    naive pattern of holding many complete deep-copied models simultaneously.'''
    d_model = model.config.hidden_size
    # Same fix as make_local_orthogonal_mlp above: build + QR on CPU, move
    # only the final model-dtype matrix to GPU.
    Q = cpu_orthogonal(d_model, seed=seed).to(device=device, dtype=model.model.embed_tokens.weight.dtype)

    param_refs, variants = [], []
    def add(module, attr, new_tensor):
        param_refs.append((module, attr)); variants.append(new_tensor)

    add(model.model.embed_tokens, 'weight', model.model.embed_tokens.weight.data @ Q)
    for layer in get_layers(model):
        add(layer.self_attn.q_proj, 'weight', layer.self_attn.q_proj.weight.data @ Q)
        add(layer.self_attn.k_proj, 'weight', layer.self_attn.k_proj.weight.data @ Q)
        add(layer.self_attn.v_proj, 'weight', layer.self_attn.v_proj.weight.data @ Q)
        add(layer.self_attn.o_proj, 'weight', Q.T @ layer.self_attn.o_proj.weight.data)
        add(layer.mlp.gate_proj, 'weight', layer.mlp.gate_proj.weight.data @ Q)
        add(layer.mlp.up_proj, 'weight', layer.mlp.up_proj.weight.data @ Q)
        add(layer.mlp.down_proj, 'weight', Q.T @ layer.mlp.down_proj.weight.data)
    if hasattr(model, 'lm_head') and not getattr(model.config, 'tie_word_embeddings', False):
        add(model.lm_head, 'weight', model.lm_head.weight.data @ Q)

    return ConditionModel(model, param_refs, variants, label='global_rotation')


def make_gaussian_noise_mlp(model, layer_idx, sigma_frac=0.1, seed=0, device='cpu'):
    mlp = get_mlp(model, layer_idx)
    torch.manual_seed(seed)
    param_refs = [(mlp.gate_proj, 'weight'), (mlp.up_proj, 'weight'), (mlp.down_proj, 'weight')]
    variants = []
    for (module, attr) in param_refs:
        w = getattr(module, attr).data
        sigma = sigma_frac * w.std()
        variants.append(w + torch.randn_like(w) * sigma)
    return ConditionModel(model, param_refs, variants, label=f'gaussian_noise_L{layer_idx}_s{sigma_frac}')


def make_neuron_deletion_mlp(model, layer_idx, delete_frac=0.1, seed=0, device='cpu'):
    mlp = get_mlp(model, layer_idx)
    d_ff = mlp.gate_proj.out_features
    torch.manual_seed(seed)
    n_delete = int(round(d_ff * delete_frac))
    delete_idx = torch.randperm(d_ff)[:n_delete]

    gate_w = mlp.gate_proj.weight.data.clone(); gate_w[delete_idx, :] = 0.0
    up_w = mlp.up_proj.weight.data.clone(); up_w[delete_idx, :] = 0.0
    down_w = mlp.down_proj.weight.data.clone(); down_w[:, delete_idx] = 0.0
    param_refs = [(mlp.gate_proj, 'weight'), (mlp.up_proj, 'weight'), (mlp.down_proj, 'weight')]
    variants = [gate_w, up_w, down_w]
    if mlp.gate_proj.bias is not None:
        b = mlp.gate_proj.bias.data.clone(); b[delete_idx] = 0.0
        param_refs.append((mlp.gate_proj, 'bias')); variants.append(b)
    if mlp.up_proj.bias is not None:
        b = mlp.up_proj.bias.data.clone(); b[delete_idx] = 0.0
        param_refs.append((mlp.up_proj, 'bias')); variants.append(b)
    return ConditionModel(model, param_refs, variants, label=f'neuron_deletion_L{layer_idx}_f{delete_frac}')


print("=" * 78)
print("SECTION 3: ALTERNATIVE PERTURBATION CONTROLS, ALL THREE FAMILIES")
print("=" * 78)
print("(Conditions are built and battery-tested ONE AT A TIME below -- rather than")
print(" materializing a dict of ~10 perturbed variants up front -- so peak memory")
print(" only ever holds the base model plus whichever single condition is active.)")

perturbation_records = {}
for name in MODEL_CONFIGS:
    m, tok = models[name], tokenizers[name]
    print(f"\n-- {name} --")
    recs = []

    recs.append(run_battery(m, tok, EVAL_CORPUS, DEVICE, label='baseline'))
    recs.append(run_battery(mlp_perm_models[name]['aligned'], tok, EVAL_CORPUS, DEVICE, label='aligned permutation'))

    local_rot = make_local_orthogonal_mlp(m, TARGET_LAYER, seed=0, device=DEVICE)
    recs.append(run_battery(local_rot, tok, EVAL_CORPUS, DEVICE, label='local rotation (d_ff)'))
    del local_rot

    # Global rotation needs ~2x THIS model's memory while it's being built (every
    # weight gets a rotated replacement computed before any swap happens -- see
    # make_global_rotation's docstring). With all three checkpoints resident, that
    # peak doesn't fit on a single GPU even after freeing local_rot. So the two
    # OTHER (currently idle) models are moved to CPU just for this one step, then
    # restored -- the only place in the notebook that needs this.
    other_names = [n for n in MODEL_CONFIGS if n != name]
    for other in other_names:
        models[other] = models[other].to('cpu')
    free_memory()
    print_gpu_mem(f"  (offloaded {other_names} to CPU for global rotation)")

    global_rot = make_global_rotation(m, seed=0, device=DEVICE)
    recs.append(run_battery(global_rot, tok, EVAL_CORPUS, DEVICE, label='global rotation (residual)'))
    del global_rot
    free_memory()  # global rotation is the one genuinely full-size condition here

    for other in other_names:
        models[other] = models[other].to(DEVICE)
    free_memory()
    print_gpu_mem(f"  (restored {other_names} to GPU)")

    for s in [0.05, 0.1, 0.25]:
        noisy = make_gaussian_noise_mlp(m, TARGET_LAYER, sigma_frac=s, seed=0, device=DEVICE)
        recs.append(run_battery(noisy, tok, EVAL_CORPUS, DEVICE, label=f'gaussian noise (sigma={s:.2f}x)'))
        del noisy

    for f in [0.1, 0.25, 0.5]:
        deleted = make_neuron_deletion_mlp(m, TARGET_LAYER, delete_frac=f, seed=0, device=DEVICE)
        recs.append(run_battery(deleted, tok, EVAL_CORPUS, DEVICE, label=f'neuron deletion ({int(f*100)}%)'))
        del deleted

    perturbation_records[name] = recs
    free_memory()

fig, axes = plt.subplots(1, len(MODEL_CONFIGS), figsize=(6.5 * len(MODEL_CONFIGS), 5), sharex=False)
for ax, name in zip(axes, MODEL_CONFIGS):
    recs = perturbation_records[name]
    labels = [r['label'] for r in recs]
    accs = [r['accuracy'] for r in recs]
    ax.barh(labels, accs, color='#4c72b0')
    ax.axvline(baseline_results[name]['accuracy'], color='gray', ls='--', lw=1)
    ax.set_title(name)
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig('perturbation_controls_all_models.png', dpi=150, bbox_inches='tight')
plt.show()
print_gpu_mem("after Section 3")


## Part 2 -- Identity boundary: graded symmetry violation

In [ ]:
def partial_violation_mlp_permutation(model, layer_idx, violation_frac, seed=0, device='cpu'):
    '''ConditionModel version (see Part 0): this sweep calls the function
    7 fracs x 5 seeds = 35 times per model, which under a naive deepcopy-per-call
    pattern would mean 35 full-model duplications per model, back to back. Here
    each call only ever allocates one layer's worth of replacement tensors.'''
    mlp = get_mlp(model, layer_idx)
    d_ff = mlp.gate_proj.out_features
    torch.manual_seed(seed)

    perm = torch.randperm(d_ff)
    inverse_perm = torch.argsort(perm)
    comp_perm = inverse_perm.clone()
    n_violate = int(round(d_ff * violation_frac))
    if n_violate >= 2:
        violate_positions = torch.randperm(d_ff)[:n_violate]
        shuffled_values = comp_perm[violate_positions][torch.randperm(n_violate)]
        comp_perm[violate_positions] = shuffled_values

    param_refs = [(mlp.gate_proj, 'weight'), (mlp.up_proj, 'weight'), (mlp.down_proj, 'weight')]
    variants = [
        mlp.gate_proj.weight.data[perm, :].clone(),
        mlp.up_proj.weight.data[perm, :].clone(),
        mlp.down_proj.weight.data[:, comp_perm].clone(),
    ]
    if mlp.gate_proj.bias is not None:
        param_refs.append((mlp.gate_proj, 'bias')); variants.append(mlp.gate_proj.bias.data[perm].clone())
    if mlp.up_proj.bias is not None:
        param_refs.append((mlp.up_proj, 'bias')); variants.append(mlp.up_proj.bias.data[perm].clone())
    return ConditionModel(model, param_refs, variants, label=f'partial_violation_L{layer_idx}_v{violation_frac}')



print("=" * 78)
print("SECTION 4: IDENTITY BOUNDARY -- GRADED SYMMETRY VIOLATION, ALL FAMILIES")
print("=" * 78)

VIOLATION_FRACS = [0.0, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
N_BOUNDARY_SEEDS = 5

boundary_records = {}
for name in MODEL_CONFIGS:
    m, tok = models[name], tokenizers[name]
    print(f"\n-- {name} --")
    recs = []
    for v in VIOLATION_FRACS:
        accs = []
        for seed in range(N_BOUNDARY_SEEDS):
            pm = partial_violation_mlp_permutation(m, TARGET_LAYER, v, seed=seed, device=DEVICE)
            accs.append(next_token_accuracy(pm, tok, EVAL_CORPUS, DEVICE))
        rec = {'violation_frac': v, 'acc_mean': float(np.mean(accs)), 'acc_std': float(np.std(accs))}
        recs.append(rec)
        print(f"  violation={v*100:5.0f}% | accuracy={rec['acc_mean']:.4f} +/- {rec['acc_std']:.4f}")
    boundary_records[name] = recs

fig, ax = plt.subplots(figsize=(7, 5))
for name in MODEL_CONFIGS:
    recs = boundary_records[name]
    v_pct = [r['violation_frac'] * 100 for r in recs]
    means = [r['acc_mean'] for r in recs]
    stds = [r['acc_std'] for r in recs]
    ax.errorbar(v_pct, means, yerr=stds, marker='o', capsize=3, label=name, color=COLORS[name])
ax.set_xlabel('Symmetry violation (%)')
ax.set_ylabel('Next-token accuracy')
ax.set_title('Identity degradation vs. violation fraction, all families')
ax.legend()
plt.tight_layout()
plt.savefig('identity_boundary_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 2 -- Representation geometry: CKA, Procrustes, principal angles

In [ ]:
def procrustes_distance(X, Y):
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    X = X / (np.linalg.norm(X) + 1e-8)
    Y = Y / (np.linalg.norm(Y) + 1e-8)
    R, _ = orthogonal_procrustes(X, Y)
    return float(np.linalg.norm(X @ R - Y))


def top_k_principal_angles(hidden_a, hidden_b, k=10):
    a = hidden_a - hidden_a.mean(axis=0, keepdims=True)
    b = hidden_b - hidden_b.mean(axis=0, keepdims=True)
    _, _, Va = np.linalg.svd(a, full_matrices=False)
    _, _, Vb = np.linalg.svd(b, full_matrices=False)
    k = min(k, Va.shape[0], Vb.shape[0])
    s = np.linalg.svd(Va[:k].T.T @ Vb[:k].T, compute_uv=False)
    s = np.clip(s, -1.0, 1.0)
    return np.degrees(np.arccos(s))


print("=" * 78)
print("SECTION 5: REPRESENTATION GEOMETRY, ALL FAMILIES")
print("=" * 78)

geometry_records = {}
for name in MODEL_CONFIGS:
    m, tok = models[name], tokenizers[name]
    conditions = {
        'aligned': mlp_perm_models[name]['aligned'],
        'unaligned': mlp_perm_models[name]['unaligned'],
        'head_aligned': head_perm_models[name]['aligned'],
        'head_unaligned': head_perm_models[name]['unaligned_nc'],
    }
    n_layers_plus_embed = len(get_layers(m)) + 1
    recs = {cn: {'cka': [], 'procrustes': [], 'angle': []} for cn in conditions}
    for layer in range(n_layers_plus_embed):
        h_base = collect_hidden_states(m, tok, EVAL_CORPUS, layer, DEVICE)
        for cn, cm in conditions.items():
            h_c = collect_hidden_states(cm, tok, EVAL_CORPUS, layer, DEVICE)
            recs[cn]['cka'].append(linear_cka(h_base, h_c))
            recs[cn]['procrustes'].append(procrustes_distance(h_base, h_c))
            recs[cn]['angle'].append(float(np.mean(top_k_principal_angles(h_base, h_c, k=10))))
    geometry_records[name] = recs
    print(f"{name}: done ({n_layers_plus_embed} layers x {len(conditions)} conditions)")

## Part 2 -- Harder tasks: does symmetry survive few-shot reasoning?

In [ ]:
def few_shot_accuracy(model, tokenizer, examples, device='cpu'):
    hits = 0
    with torch.no_grad():
        for prompt, target in examples:
            ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
            target_id = tokenizer.encode(target, add_special_tokens=False)[0]
            pred_id = model(ids).logits[0, -1, :].argmax(dim=-1).item()
            hits += int(pred_id == target_id)
    return hits / len(examples)


print("=" * 78)
print("SECTION 6: HARDER-TASK SURVIVAL -- FEW-SHOT ANTONYM COMPLETION")
print("=" * 78)

for name in MODEL_CONFIGS:
    tok = tokenizers[name]
    for cond, m in [('baseline', models[name]), ('aligned', mlp_perm_models[name]['aligned']),
                     ('unaligned', mlp_perm_models[name]['unaligned'])]:
        acc = few_shot_accuracy(m, tok, FEW_SHOT_ANTONYM, DEVICE)
        print(f"{name:>16s} / {cond:>10s}: few-shot antonym top-1 accuracy = {acc:.3f}")

## Part 2 -- Stats and summary

In [ ]:
print("=" * 78)
print("SECTION 7: SIGNIFICANCE TESTS (paired, over multi-seed sweep) + SUMMARY")
print("=" * 78)

for name in MODEL_CONFIGS:
    rec = seed_sweep_records[name]
    t_stat, p_val = scipy_stats.ttest_rel(rec['a_diffs'], rec['u_diffs'])
    print(f"{name}: paired t-test aligned-vs-unaligned accuracy diffs, "
          f"t={t_stat:.2f}, p={p_val:.2e}")

print("\nSummary table (accuracy delta from baseline, layer", TARGET_LAYER, "):")
print(f"{'model':>16s} {'aligned':>10s} {'unaligned':>10s} {'gate-only':>10s}")
for name in MODEL_CONFIGS:
    b = mlp_perm_battery[name]
    print(f"{name:>16s} {b['aligned']['accuracy']-b['baseline']['accuracy']:+10.4f} "
          f"{b['unaligned']['accuracy']-b['baseline']['accuracy']:+10.4f} "
          f"{b['gateonly']['accuracy']-b['baseline']['accuracy']:+10.4f}")

## Part 2 (continued, added) -- methodological refinement (Point 5)

Four additions, computed here from the numbers already produced in the sections above (no new
model runs needed): relative performance drop, an analytic chance-level control, effect sizes
(Hedges' g / Cohen's d) for the aligned-vs-unaligned comparison, and an inter-model ANOVA with
Bonferroni-corrected pairwise follow-ups.

For the effect-size and ANOVA cells, the seed-level draws behind the multi-seed sweep (Part 1,
STEP 8) were not individually persisted to the printed output, only their mean and 95% CI (N=15).
The cells below recover an implied per-seed standard deviation from the reported CI under a
t-distribution with df=14 (`SD = half-width * sqrt(N) / t_crit`) and are explicit about doing so --
a transparent reconstruction from reported summary statistics, not a rerun of the original seeds.
If raw per-seed values are exported from the sweep (a one-line change: append each seed's
`acc_diff` to a list instead of only accumulating mean/CI), swap that array in directly and
everything downstream is unaffected.


In [ ]:
# (a) Relative performance drop, all conditions where a same-model baseline exists
baseline_acc  = {'Qwen2.5-1.5B': 0.4650, 'Gemma-2-2b': 0.4655, 'Llama-3.2-1B': 0.4292}
conditions = {
    'unaligned (layer 4, full battery)':  {'Qwen2.5-1.5B': 0.3696, 'Gemma-2-2b': 0.4519, 'Llama-3.2-1B': 0.4201},
    'gate-only (layer 4, full battery)':  {'Qwen2.5-1.5B': 0.4023, 'Gemma-2-2b': 0.4417, 'Llama-3.2-1B': 0.3592},
    'local rotation (theta=1, single-shot)': {'Qwen2.5-1.5B': 0.0114, 'Gemma-2-2b': 0.4655, 'Llama-3.2-1B': 0.2647},
    'global rotation (residual stream)':  {'Qwen2.5-1.5B': 0.0000, 'Gemma-2-2b': 0.0968, 'Llama-3.2-1B': 0.0000},
    'neuron deletion 50%':                {'Qwen2.5-1.5B': 0.4329, 'Gemma-2-2b': 0.4530, 'Llama-3.2-1B': 0.4428},
}

print(f"{'condition':38} | {'Qwen2.5-1.5B':>14} | {'Gemma-2-2b':>12} | {'Llama-3.2-1B':>14}")
for cond, accs in conditions.items():
    row = [f"{(baseline_acc[n]-accs[n])/baseline_acc[n]:+.1%}" for n in baseline_acc]
    print(f"{cond:38} | {row[0]:>14} | {row[1]:>12} | {row[2]:>14}")


**Reading the relative-drop table.** Framing everything as a percentage of each model's own
baseline is what makes the Qwen/Gemma contrast comparable at all: Qwen's 0.4650->0.0114 and
Llama's 0.4292->0.2647 aren't directly comparable as raw deltas (different baselines), but as
relative drops (97.5% vs. 38.3%) they're on the same 0-100% scale. Note the ordering isn't
consistent across conditions -- Llama has the *largest* relative drop under gate-only permutation
(16.3%, vs. Qwen's 13.5%) despite being more robust than Qwen under rotation -- which is the point
made again after the ANOVA below: "robustness" isn't one number per model.


In [ ]:
# (b) Analytic chance-level (random-vector) control: next-token top-1 accuracy under a
# uniform-random categorical over the vocabulary is exactly 1/|V| in expectation -- no
# model run needed, only each model's vocabulary size (already in its config above).
vocab = {'Qwen2.5-1.5B': 151936, 'Gemma-2-2b': 256128, 'Llama-3.2-1B': 128256}

print("Chance / random-vector baseline (analytic):")
for name, V in vocab.items():
    print(f"  {name:>15}: chance accuracy = 1/{V} = {1/V:.2e}   chance perplexity = {V}")

print()
print("How far each already-reported condition sits above the chance floor (multiples of chance):")
baseline_acc  = {'Qwen2.5-1.5B': 0.4650, 'Gemma-2-2b': 0.4655, 'Llama-3.2-1B': 0.4292}
unaligned_acc = {'Qwen2.5-1.5B': 0.3696, 'Gemma-2-2b': 0.4519, 'Llama-3.2-1B': 0.4201}
local_rot_acc = {'Qwen2.5-1.5B': 0.0114, 'Gemma-2-2b': 0.4655, 'Llama-3.2-1B': 0.2647}
for name in baseline_acc:
    V = vocab[name]; chance = 1 / V
    print(f"  {name:>15}: baseline={baseline_acc[name]/chance:>9,.0f}x chance | "
          f"unaligned={unaligned_acc[name]/chance:>9,.0f}x chance | "
          f"local-rotation={local_rot_acc[name]/chance:>9,.0f}x chance")
print()
print("Even Qwen's near-total collapse under local rotation (0.0114) is still ~1,700x the")
print("analytic chance floor -- the model hasn't been reduced to a random-vector baseline, it")
print("has lost the specific structure that gave it the other ~40 points of accuracy.")


In [ ]:
# (c) Effect sizes for the aligned-vs-unaligned seed sweep -- computed directly from the
# REAL per-seed arrays already sitting in `seed_sweep_records` (STEP 8 / cell above, N=15
# actual forward-pass results per model), instead of reconstructing a standard deviation
# from a previously-reported confidence interval.
import numpy as np
from scipy import stats as st

N = N_SEEDS  # =15, defined at STEP 8 above
hedges_corr = 1 - 3 / (4 * (N - 1) - 1)

print("Effect sizes (one-sample, against a null of zero accuracy change), N=15, real per-seed data:")
print(f"{'model':>15} | {'mean':>8} | {'sample SD':>9} | {'Cohen d':>9} | {'Hedges g':>9}")
recon = {}
for name in MODEL_CONFIGS:
    diffs = seed_sweep_records[name]['u_diffs']       # real array, length N, from STEP 8
    mean, sd = float(diffs.mean()), float(diffs.std(ddof=1))
    cohend = mean / sd
    hedgesg = cohend * hedges_corr
    recon[name] = dict(mean=mean, sd=sd, diffs=diffs)
    print(f"{name:>15} | {mean:8.4f} | {sd:9.4f} | {cohend:9.3f} | {hedgesg:9.3f}")


**Reading the effect sizes.** Watch for two things once this runs on real per-seed data: (1)
whether all three |g| values clear the conventional "large effect" threshold of 0.8 -- if so, the
aligned/unaligned distinction isn't in question for any model, only its magnitude is; (2) whether
the effect-size ordering across models matches the accuracy-drop ordering already visible in the
raw numbers above. These are now computed straight from the N=15 real per-seed arrays in
`seed_sweep_records`, not a standard deviation reconstructed from a reported confidence interval.


In [ ]:
# (d) Inter-model comparison: one-way ANOVA + Bonferroni-corrected pairwise Welch t-tests,
# on the REAL per-seed samples from STEP 8 (`seed_sweep_records[name][\'u_diffs\']`). The
# previous version of this cell drew np.random.normal(mean, sd) "synthetic per-seed samples"
# from a reconstructed summary statistic -- that's a legitimate parametric-bootstrap technique
# in general, but it throws away the actual N=15 measurements already sitting in memory from
# STEP 8, so it's replaced here with the real arrays directly.
samples = {name: recon[name]['diffs'] for name in MODEL_CONFIGS}

f_stat, p_anova = st.f_oneway(*samples.values())
print(f"One-way ANOVA across models (unaligned accuracy-diff, N=15/model, real data): "
      f"F={f_stat:.3f}, p={p_anova:.4g}")

names = list(samples.keys())
pairs = [(names[i], names[j]) for i in range(len(names)) for j in range(i + 1, len(names))]
alpha = 0.05
bonf = alpha / len(pairs)
print(f"\nPairwise Welch t-tests, Bonferroni-corrected alpha = {alpha}/{len(pairs)} = {bonf:.4f}")
for a, b in pairs:
    t, p = st.ttest_ind(samples[a], samples[b], equal_var=False)
    sig = "significant" if p < bonf else "not significant"
    print(f"  {a} vs {b}: t={t:.3f}, p={p:.4g}  -> {sig} after Bonferroni")


**Reading the ANOVA.** This cell now runs on the real N=15-per-model arrays rather than a
resampled approximation, so the exact F and p values (and which pairwise comparisons clear the
Bonferroni-corrected threshold) should be read off this run's actual printed output, not assumed
in advance. What's worth checking specifically: whether the pairwise pattern for this perturbation
(MLP permutation, seed-swept) matches or diverges from the rotation-sweep story in Part 4 below
(Qwen collapses and stays down, Llama dips then partially recovers) -- if the two perturbations
rank the models differently, that's evidence the models aren't uniformly "robust" or "fragile" in
a way that transfers across perturbation types, which is exactly the kind of overclaim relative
drop and inter-model testing, taken together, are meant to catch.


## Part 3 -- Generalization: scaling, cross-family architecture, training-time, cross-model

**Motivating literature.** Same ethos as the original's Part 3: is the exactness result
an artifact of one model/scale, or does it generalize? Here "scaling" and "cross-arch"
are answered by the three real checkpoints themselves (1.24B / 1.5B / 2.6B params;
different GQA ratios; SiLU vs. GeGLU activations; Gemma-2's alternating sliding-window
attention) rather than by synthetic scale variants -- a stronger test than the original
could run with only GPT-2 sizes, since these are architecturally distinct models, not
just wider/narrower versions of the same design.

In [ ]:
print("=" * 78)
print("PART 3, STEP A: SCALING -- DOES EXACTNESS DEPEND ON MODEL SCALE?")
print("=" * 78)

scale_table = []
for name, cfg in MODEL_CONFIGS.items():
    m = models[name]
    n_params = sum(p.numel() for p in m.parameters())
    b = mlp_perm_battery[name]
    scale_table.append(dict(
        name=name, n_params=n_params,
        aligned_acc_delta=b['aligned']['accuracy'] - b['baseline']['accuracy'],
        unaligned_acc_delta=b['unaligned']['accuracy'] - b['baseline']['accuracy'],
    ))
    print(f"{name:>16s}: {n_params/1e9:.2f}B params | "
          f"aligned delta={scale_table[-1]['aligned_acc_delta']:+.5f}  "
          f"unaligned delta={scale_table[-1]['unaligned_acc_delta']:+.5f}")

print("\nExpected: aligned delta ~0 REGARDLESS of scale (it's an exact algebraic")
print("identity, not a statistical property that could wash out or emerge with scale).")

In [ ]:
print("=" * 78)
print("PART 3, STEP B: CROSS-FAMILY ARCHITECTURE DIFFERENCES")
print("=" * 78)

for name in MODEL_CONFIGS:
    m = models[name]
    cfg = m.config
    act = getattr(cfg, 'hidden_act', 'unknown')
    n_heads, n_kv = get_num_heads(m), get_num_kv_heads(m)
    sliding = getattr(cfg, 'sliding_window', None)
    print(f"{name:>16s}: activation={act:>14s}  GQA ratio={n_heads}:{n_kv}  "
          f"sliding_window={sliding}")

print("\nGemma-2 alternates full and sliding-window attention across layers (see its")
print("config's `sliding_window` / layer_types). If TARGET_LAYER for Gemma-2 falls on a")
print("sliding-window layer, the permutation math above is UNCHANGED (windowing affects")
print("which keys are attended to, not the linear-algebra structure of Q/K/V/O), but")
print("it's worth re-running STEP 2 with TARGET_LAYER set to both a full-attention and a")
print("sliding-window layer index to confirm empirically -- check")
print("`models['Gemma-2-2b'].config` for the pattern before picking layers.")

## Part 3 -- Training-time: does a few steps of fine-tuning preserve the symmetry?

**Motivating literature.** This section has no direct analogue in the cited
mode-connectivity papers (which compare *independently trained* endpoints); it's this
project's own check of a basic fact about SGD: since permutation is a linear
reindexing, and gradients of a permutation-equivariant loss w.r.t. permuted weights are
themselves permuted the same way, aligned-permuted and baseline weights should receive
correspondingly-permuted gradients and stay exactly equivalent after any number of full
(non-stochastic-index) optimizer steps, up to floating point error. This is a real,
checkable claim, not an assumption -- the cell below fine-tunes both the baseline and
its aligned-permutation twin for a few steps and re-checks exactness.

**Implementation note.** The aligned twin is built with a full, real permutation of its
own weights (`make_aligned_mlp_permutation_real`), not the `ConditionModel` swap used
elsewhere in this notebook. `ConditionModel` sets `.data = variant` for a forward call
and restores `.data = original` in a `finally` block that runs *before* the training
loop's `backward()`/`opt.step()`. The gradient itself still comes out correct (PyTorch's
saved-tensors capture the value actually used in forward, not a live read through
`.data`), but the optimizer step lands on the just-restored *original* weight -- i.e. a
permuted-order gradient applied to unpermuted-order storage. That's a coordinate-basis
mismatch, not a valid SGD step, so it would have silently corrupted the aligned twin's
training and made this cell's comparison meaningless. Baking the permutation into real,
persisted weights on its own full copy avoids that: gradients and updates for the
aligned twin live in the same basis throughout.

### Fix (added) -- a transient-memory bug in the fine-tuning check above

`ft_baseline = copy.deepcopy(m)` (and the identical pattern inside
`make_aligned_mlp_permutation_real` above) deep-copies the active model **while the original is
still resident on GPU**. For a moment both the original and its full copy are on the card at
once -- a transient ~2x spike. Qwen2.5-1.5B and Llama-3.2-1B's smaller weights (~3.1GB, ~2.5GB)
absorb that fine. Gemma-2-2b's ~5.2GB weights do not: that transient alone is ~10.4GB, and once
`finetune_in_place` allocates its own ~5.2GB of gradient buffers plus backward-pass activations
on top of whatever fragmentation the many earlier offload/restore cycles left behind, the card
has nothing left. (Symptom if unpatched: `OutOfMemoryError` specifically on Gemma-2-2b's
iteration of the loop below, after Qwen's iteration already succeeded -- same code, larger
model.)

Fixed in the cell below: evict the original to CPU **before** deep-copying it, in both places
this pattern occurs, so the copy is always made CPU-to-CPU and only one full copy of the model
is ever on GPU at a time. Peak GPU use for Gemma-2-2b's fine-tune drops from ~15.6GB+ (two
copies' weights + one copy's gradients) to ~10.4GB (one copy's weights + its own gradients).

**Second fix (confirmed by a real Kaggle T4x2 run of this notebook -- see the traceback this
revision is responding to):** `make_aligned_mlp_permutation_real` accepted a `device` argument
but never used it, so its returned deep copy stayed on CPU. `finetune_in_place` only moves the
*token ids* to `device`, not the model, so the very first training step crashed with `Expected
all tensors to be on the same device, but got index is on cuda:0, different from other tensors
on cpu` -- Qwen's baseline branch (which does `.to(DEVICE)` explicitly on its own copy)
completed fine before this happened on the very next cell, `aligned_before_ft`. Fixed at the
source: the function now moves its own returned copy to `device` before returning it, so the
`device=DEVICE` every call site already passes is actually honored.


In [ ]:
import torch.nn.functional as F

def make_aligned_mlp_permutation_real(model, layer_idx, seed=0, device='cpu'):
    '''Like make_aligned_mlp_permutation (Part 2), but bakes the permutation into
    REAL, persisted weights on a full model copy instead of a transient
    ConditionModel swap. This is required for training: ConditionModel's
    swap-forward-restore pattern sets `.data = variant` for the forward pass and
    restores `.data = original` in a `finally` block that runs BEFORE backward()
    is called from the training loop. backward() itself still computes gradients
    correctly with respect to the variant that was actually used in forward (this
    was checked empirically -- PyTorch's saved-tensors are a snapshot, not a live
    read through `.data`), but by the time `opt.step()` runs, `.data` has already
    been reset to the ORIGINAL weight. That means the update becomes
    `original_weight -= lr * grad_of_permuted_variant`: a permuted-order gradient
    landing on unpermuted-order storage, which is a coordinate-basis mismatch, not
    a valid SGD step. Baking the permutation in for real avoids this: the copy's
    weights and its own gradients live in the SAME (permuted) basis throughout
    training, so ordinary SGD is valid, and the aligned copy is a genuine
    standalone model at eval time (no stale/frozen swap-in required).

    Same permutation convention as make_aligned_mlp_permutation: gate_proj and
    up_proj output rows permuted together, down_proj input columns compensated by
    the same permutation, so the function computed by the MLP is unchanged before
    any fine-tuning happens.

    Memory note: this moves `model` to CPU before deep-copying it, so the copy is made
    CPU-to-CPU rather than leaving the original and its full copy briefly co-resident on
    GPU (a transient ~2x spike that's enough to OOM a ~2.6B-parameter model on a 15GB
    card once combined with the training copy's own gradient buffers a few lines below).
    `model` is left on CPU when this returns -- move it back to `device` yourself if you
    still need it there (the call site below doesn't; it evicts `model` right after
    calling this anyway).
    '''
    model.to('cpu')                      # avoid a transient 2x-on-GPU spike during deepcopy
    m = copy.deepcopy(model)
    mlp = get_mlp(m, layer_idx)
    torch.manual_seed(seed)
    perm = torch.randperm(mlp.gate_proj.out_features)
    with torch.no_grad():
        mlp.gate_proj.weight.data = mlp.gate_proj.weight.data[perm, :].clone()
        mlp.up_proj.weight.data = mlp.up_proj.weight.data[perm, :].clone()
        mlp.down_proj.weight.data = mlp.down_proj.weight.data[:, perm].clone()
        if mlp.gate_proj.bias is not None:
            mlp.gate_proj.bias.data = mlp.gate_proj.bias.data[perm].clone()
        if mlp.up_proj.bias is not None:
            mlp.up_proj.bias.data = mlp.up_proj.bias.data[perm].clone()
        # down_proj.bias (if present) lives on the OUTPUT dim (hidden_size), not
        # d_ff, so it's untouched by this permutation -- same as the ConditionModel
        # version in Part 2.
    # BUG FIX (confirmed by a real Kaggle run): `device` was accepted as a parameter but
    # never actually used inside this function -- the deep copy `m` inherits CPU placement
    # from `model` (moved to CPU a few lines above) and was returned as-is, still on CPU.
    # The call site then ran `finetune_in_place(aligned_before_ft, ..., device=DEVICE)`,
    # which only moves the training text's token ids to DEVICE, not the model itself, so
    # the embedding lookup crashed with "Expected all tensors to be on the same device,
    # but got index is on cuda:0, different from other tensors on cpu" the instant training
    # started. The baseline branch a few lines below avoids this by chaining `.to(DEVICE)`
    # onto its own deepcopy explicitly; this function needs the same fix applied
    # internally, since every caller passes device=DEVICE expecting it to be honored.
    m = m.to(device)
    return m, perm


def finetune_in_place(model, tokenizer, texts, n_steps=5, lr=1e-5, device='cpu'):
    '''Trains `model` in place with ordinary SGD and returns it. Never wraps a
    ConditionModel (see make_aligned_mlp_permutation_real's docstring for why
    that's unsafe to combine with backward()+step()). The caller owns the
    model's lifetime: create your copy right before calling this, and
    `del` it + call free_memory() as soon as you're done comparing it, so
    at most one extra full-model copy is ever alive at a time -- important
    on a 16GB T4, where the three base checkpoints alone already use most of
    the fp16 budget (see PART 3, STEP C's memory note below).'''
    for p in model.parameters():
        p.requires_grad_(True)  # base models are loaded with requires_grad_(False)
    model.train()
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    for step in range(n_steps):
        opt.zero_grad()
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            out = model(ids, labels=ids)
            out.loss.backward()
        opt.step()
    opt.zero_grad(set_to_none=True)  # drop the last step's .grad buffers now
    model.eval()                     # rather than waiting on del+gc to reclaim them
    for p in model.parameters():
        p.requires_grad_(False)
    return model


print("=" * 78)
print("PART 3, STEP C: TRAINING-TIME SYMMETRY PRESERVATION")
print("=" * 78)
print("Memory budget for this cell (why the schedule below looks the way it does):")
print("  - the two IDLE checkpoints are offloaded to CPU for the whole loop body,")
print("    same fix as the global-rotation step in Section 3 -- they contribute")
print("    ~0GB to GPU while a third model is under test.")
print("  - within that, the ACTIVE model m is itself only needed on GPU at the two")
print("    instants it gets deep-copied (start of baseline, start of aligned); it")
print("    contributes nothing to the forward/backward passes that follow. Parking")
print("    it on CPU for the duration of each fine-tune -- not just deleting the")
print("    finished copy afterward -- caps sustained peak GPU use at ~2x the active")
print("    model's weight size (training copy + its own .grad buffers), instead of")
print("    ~3x (m + copy weights + copy grad) if m stayed resident throughout.")
print("    For Gemma-2-2b (~5.2GB/copy) that's the difference between ~10.4GB and")
print("    ~15.7GB against this T4's ~14.6GB usable ceiling -- i.e. the difference")
print("    between comfortable headroom and a guaranteed OOM.")
print_gpu_mem("before fine-tuning checks")

for name in MODEL_CONFIGS:
    m, tok = models[name], tokenizers[name]
    print(f"\n-- {name} --")

    other_names = [n for n in MODEL_CONFIGS if n != name]
    for other in other_names:
        models[other] = models[other].to('cpu')
    free_memory()
    print_gpu_mem(f"  (offloaded {other_names} to CPU for fine-tuning check)")

    sample_ids = tok(EVAL_CORPUS[0], return_tensors='pt').input_ids.to(DEVICE)

    # --- fine-tuned baseline ---
    # Evict the original to CPU BEFORE deep-copying it, so the copy is made CPU-to-CPU
    # and only ONE full copy of the model is ever resident on GPU at a time (see the
    # memory note on make_aligned_mlp_permutation_real above for the full diagnosis --
    # the previous version deep-copied while `m` was still on GPU, which OOM'd on
    # Gemma-2-2b's larger weights).
    m = models[name] = m.to('cpu')          # evict the original now -- not needed again until
    free_memory()                           # the aligned copy is built below
    ft_baseline = copy.deepcopy(m).to(DEVICE)
    free_memory()
    try:
        finetune_in_place(ft_baseline, tok, EVAL_CORPUS[:4], n_steps=3, lr=1e-5, device=DEVICE)
    except torch.cuda.OutOfMemoryError:
        print_gpu_mem(f"  OOM during {name} baseline fine-tune")
        raise RuntimeError(
            f"OOM fine-tuning {name}'s baseline copy even after offloading the other "
            f"two checkpoints and evicting the original -- this shouldn't happen at "
            f"the current EVAL_CORPUS[:4]/n_steps=3 budget on a ~14.6GB T4. If you've "
            f"changed EVAL_CORPUS to longer texts or grown n_steps, shrink one of those "
            f"for this model, or switch PRECISION_MODE to 'bf16'/'fp16' if not already."
        ) from None
    with torch.no_grad():
        baseline_logits = ft_baseline(sample_ids).logits.clone()
    del ft_baseline
    free_memory()
    print_gpu_mem(f"after baseline fine-tuning check for {name}")

    # --- fine-tuned aligned (real permutation, not a ConditionModel swap) ---
    # `m` is already on CPU here (evicted at the end of the baseline branch above) -- no
    # need to round-trip it back to GPU just to permute its weights; permutation is pure
    # tensor indexing and works fine on CPU. Skips a second transient-2x-on-GPU moment on
    # top of the one already fixed inside make_aligned_mlp_permutation_real itself.
    aligned_before_ft, perm = make_aligned_mlp_permutation_real(m, TARGET_LAYER, seed=0, device=DEVICE)
    free_memory()
    try:
        finetune_in_place(aligned_before_ft, tok, EVAL_CORPUS[:4], n_steps=3, lr=1e-5, device=DEVICE)
    except torch.cuda.OutOfMemoryError:
        print_gpu_mem(f"  OOM during {name} aligned fine-tune")
        raise RuntimeError(
            f"OOM fine-tuning {name}'s aligned copy -- see the baseline branch's error "
            f"message above for the same diagnosis/remedy."
        ) from None
    with torch.no_grad():
        aligned_logits = aligned_before_ft(sample_ids).logits
    del aligned_before_ft
    free_memory()
    print_gpu_mem(f"after aligned fine-tuning check for {name}")

    diff = (baseline_logits - aligned_logits).abs().max().item()
    print(f"  max|logit diff| between fine-tuned-baseline and fine-tuned-(pre-permuted-aligned): {diff:.2e}")
    print("  (expect small at this dtype's noise floor: permutation commutes with full-batch SGD steps)")

    # Restore m (evicted twice above) and the two idle checkpoints before the
    # next iteration / downstream cells, all of which expect every model on GPU.
    m = models[name] = m.to(DEVICE)
    for other in other_names:
        models[other] = models[other].to(DEVICE)
    free_memory()
    print_gpu_mem(f"after restoring all three models to GPU")


## Part 3 -- Cross-model representational similarity

This repurposes linear CKA for its *original* use case in Kornblith et al. (2019) --
comparing representations **across independently trained networks** -- rather than the
within-model falsification use above. CKA's formula is well-defined between activation
matrices of different feature dimensionality (hidden sizes differ across the three
families), so no projection or dimensionality matching is needed.

In [ ]:
print("=" * 78)
print("PART 3, STEP D: CROSS-MODEL REPRESENTATIONAL SIMILARITY (DIFFERENT FAMILIES)")
print("=" * 78)

def collect_pooled_hidden_states(model, tokenizer, texts, layer, device='cpu'):
    """Mean-pool each text's token dimension down to a single row before returning.

    BUG FIX (was: reusing collect_hidden_states here, which concatenates every
    token's hidden state into one big (total_tokens, hidden) matrix). That is
    correct for Step 5 / Section 5 above because those compare the SAME model
    against itself (baseline vs. a perturbed copy) under the SAME tokenizer, so
    the token axis is guaranteed to line up 1:1, token-for-token, across the two
    sides of the comparison.

    It is NOT valid here: Qwen/Gemma/Llama use three different tokenizers, so
    the same EVAL_CORPUS text produces a different number of tokens for each
    model (this is exactly what crashed with "size 91 is different from 85" --
    two families' concatenated token axes just don't have the same length, and
    even where lengths coincidentally matched, token i in one model's BPE
    vocabulary is not token i in another's, so no meaningful correspondence
    would exist to average over token-for-token anyway).

    The fix operates on a representation that IS well-defined across tokenizers:
    one pooled vector per example. Mean-pooling over the (model-specific-length)
    token axis collapses every text to a single hidden_dim vector, so the row
    axis becomes "which of the 8 EVAL_CORPUS texts" -- identical across all
    three families by construction -- while the column axis (hidden_dim) is
    allowed to differ freely between models, which linear_cka already supports.
    This is also strictly cheaper on GPU/host memory than the old per-token
    concatenation: output is (n_texts, hidden) instead of (total_tokens, hidden),
    and nothing extra is held on GPU -- pooling happens on the same per-text
    numpy array right after it's moved to CPU, with no extra forward passes.
    """
    rows = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            out = model(ids, output_hidden_states=True)
            h = out.hidden_states[layer][0].float().cpu().numpy()  # (seq_len, hidden)
            rows.append(h.mean(axis=0))  # pool over tokens -> (hidden,)
    return np.stack(rows, axis=0)  # (n_texts, hidden) -- n_texts matches across families


names = list(MODEL_CONFIGS.keys())
# Compare the FINAL hidden state (most semantically aggregated) across families,
# since layer *counts* differ and per-layer alignment across architectures isn't
# meaningful the way it is for the same-model-different-permutation comparisons above.
# Uses collect_pooled_hidden_states (not collect_hidden_states) so the row axis is
# "example index" rather than "token index" -- see the docstring above for why the
# per-token version is only valid for same-tokenizer comparisons.
final_hidden = {}
for name in names:
    m, tok = models[name], tokenizers[name]
    n_layers_plus_embed = len(get_layers(m)) + 1
    final_hidden[name] = collect_pooled_hidden_states(m, tok, EVAL_CORPUS, n_layers_plus_embed - 1, DEVICE)

print("Cross-family linear CKA on final hidden states (same input text, different models,")
print("each text mean-pooled over its own tokenizer's token axis -- see fix note above):")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        cka = linear_cka(final_hidden[names[i]], final_hidden[names[j]])
        print(f"  CKA({names[i]}, {names[j]}) = {cka:.4f}")

print("\nUnlike the within-model aligned-permutation case, there is no reason to expect")
print("these values near 1.0 -- independently trained models with different objectives,")
print("data, and architectures are not required to converge to similar representational")
print("geometry. Any elevated similarity here is an empirical finding about convergent")
print("representation learning (Kornblith et al.'s original question), not a symmetry check.")


## Part 3 -- Long-context evaluation

In [ ]:
LONG_CONTEXT_TEXT = (
    "The history of Rome spans over a thousand years, beginning as a small "
    "settlement on the Italian peninsula and growing into one of the largest "
    "empires in the ancient world. " * 20
).strip()

print("=" * 78)
print("PART 3, STEP E: LONG-CONTEXT EVALUATION")
print("=" * 78)

for name in MODEL_CONFIGS:
    tok = tokenizers[name]
    ids = tok(LONG_CONTEXT_TEXT, return_tensors='pt').input_ids.to(DEVICE)
    n_tokens = ids.shape[1]
    print(f"\n-- {name} ({n_tokens} tokens) --")
    for cond, m in [('baseline', models[name]), ('aligned', mlp_perm_models[name]['aligned']),
                     ('unaligned', mlp_perm_models[name]['unaligned'])]:
        with torch.no_grad():
            out = m(ids, labels=ids)
        ppl = math.exp(out.loss.item())
        print(f"  {cond:>10s}: perplexity over full long context = {ppl:9.3f}")

## Part 3 -- Annex: robust head-count / KV-head detection

Generalizes the original's GPT-NeoX/Pythia annex. Newer HF configs are inconsistent
about where head-count metadata lives; this helper tries the documented attributes in
order of specificity and falls back to inference from `hidden_size`.

In [ ]:
def robust_head_config(model):
    cfg = model.config
    n_heads = getattr(cfg, 'num_attention_heads', None)
    n_kv = getattr(cfg, 'num_key_value_heads', None)
    head_dim = getattr(cfg, 'head_dim', None)

    if n_heads is None:
        raise ValueError("Could not find num_attention_heads on this config.")
    if n_kv is None:
        n_kv = n_heads  # MHA fallback
    if head_dim is None:
        head_dim = cfg.hidden_size // n_heads

    assert n_heads % n_kv == 0, f"num_attention_heads ({n_heads}) not divisible by num_key_value_heads ({n_kv})"
    return dict(num_heads=n_heads, num_kv_heads=n_kv, head_dim=head_dim, group_size=n_heads // n_kv)


for name in MODEL_CONFIGS:
    print(name, robust_head_config(models[name]))

## Part 4 -- Identity boundary theory: continuous rotation angle sweep

Extends Part 2's discrete violation-fraction sweep with a **continuous** version:
instead of "how many neurons get shuffled," sweep the rotation angle of a 2-plane
rotation interpolating from identity ($\theta=0$) to a full orthogonal draw ($\theta$
large), tracing out exactly where behavior starts to degrade as a continuous function
rather than a discrete one. This is possible here in a way it wasn't natural for GPT-2's
single-projection MLP, because the two-branch SwiGLU/GeGLU structure gives a richer
space of partial transformations to interpolate through.

In [ ]:
def rotation_angle_sweep_mlp(model, layer_idx, angles, seed=0, device='cpu'):
    '''Generator version -- yields (theta, ConditionModel) ONE AT A TIME so the
    caller can evaluate and drop each variant before the next is built, the
    same "one condition at a time" discipline Section 3 already uses.

    This is also what fixes the OOM the previous run hit here: the old version
    (a) built A, Q_full, and I as full fp32 d_ff x d_ff buffers directly ON
    THE GPU (cuSOLVER's QR only supports fp32/fp64, so that was already the
    largest per-call allocation in the whole notebook), and (b) returned a
    *list* of all len(angles) ConditionModels up front, so all of their
    variant tensors -- plus every theta's transient fp32 blend/Qb buffer --
    stayed reachable simultaneously. With three ~1.2-2.6B checkpoints already
    resident (~11.7GB/14.6GB used), that combination was enough to OOM on
    Gemma-2-2b's larger d_ff=9216. Here, Q_full/I are built once on CPU
    (cheap, and cuSOLVER's fp32-only restriction is moot there), and only the
    single current theta's small model-dtype Qb and its resulting per-layer
    variant tensors ever reach the GPU -- freed as soon as the caller moves on
    to the next theta.
    '''
    mlp_ref = get_mlp(model, layer_idx)
    d_ff = mlp_ref.gate_proj.out_features
    model_dtype = mlp_ref.gate_proj.weight.dtype

    Q_full = cpu_orthogonal(d_ff, seed=seed)
    I = torch.eye(d_ff, dtype=torch.float32)

    for theta in angles:
        # interpolate rotation generator in the Lie algebra sense: exponential map of
        # theta * logm(Q_full) would be ideal, but for a lightweight sweep we linearly
        # blend Q_full and I then re-orthogonalize -- a simple, inspectable proxy for
        # "how much rotation." Blend + re-QR both run on CPU; only the resulting
        # matrix, already cast to model_dtype (half the fp32 footprint), is moved
        # to `device` -- and only for the moment it's needed below.
        blend = (1 - theta) * I + theta * Q_full
        Qb_cpu, _ = torch.linalg.qr(blend)
        del blend
        Qb = Qb_cpu.to(device=device, dtype=model_dtype)
        del Qb_cpu
        mlp = get_mlp(model, layer_idx)
        param_refs = [(mlp.gate_proj, 'weight'), (mlp.up_proj, 'weight'), (mlp.down_proj, 'weight')]
        variants = [
            Qb.T @ mlp.gate_proj.weight.data,
            Qb.T @ mlp.up_proj.weight.data,
            mlp.down_proj.weight.data @ Qb,
        ]
        del Qb
        yield theta, ConditionModel(model, param_refs, variants, label=f'rotation_theta{theta}_L{layer_idx}')


print("=" * 78)
print("PART 4: CONTINUOUS ROTATION-ANGLE SWEEP, ALL FAMILIES")
print("=" * 78)

ANGLES = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]
rotation_sweep_records = {}
for name in MODEL_CONFIGS:
    m, tok = models[name], tokenizers[name]
    print(f"\n-- {name} --")
    recs = []
    with cuda_oom_guard(f"PART 4 rotation sweep / {name}"):
        for theta, sm in rotation_angle_sweep_mlp(m, TARGET_LAYER, ANGLES, seed=0, device=DEVICE):
            acc = next_token_accuracy(sm, tok, EVAL_CORPUS, DEVICE)
            recs.append((theta, acc))
            print(f"  theta={theta:.2f}: accuracy={acc:.4f}")
            del sm  # drop this theta's variant tensors before the generator builds the next
    rotation_sweep_records[name] = recs
    free_memory()
    print_gpu_mem(f"after rotation sweep for {name}")

fig, ax = plt.subplots(figsize=(7, 5))
for name in MODEL_CONFIGS:
    recs = rotation_sweep_records[name]
    ax.plot([r[0] for r in recs], [r[1] for r in recs], marker='o', label=name, color=COLORS[name])
ax.set_xlabel('Interpolation parameter theta (0=identity, 1=full random rotation)')
ax.set_ylabel('Next-token accuracy')
ax.set_title('Continuous degradation under partial rotation')
ax.legend()
plt.tight_layout()
plt.savefig('rotation_angle_sweep_all_models.png', dpi=150, bbox_inches='tight')
plt.show()


## Part 4 (continued, added) -- why does Qwen collapse, Gemma shrug it off, and Llama do neither cleanly?

The sweep just above already contains the two findings that need follow-up:

* **Qwen vs. Gemma.** Qwen's accuracy is driven to floor by theta=0.10 and *stays* there through
  theta=1.00. Gemma's accuracy never leaves a narrow band (0.40-0.47) across the entire sweep -- a
  full random rotation of one layer's SwiGLU/GeGLU hidden axis is, behaviorally, almost nothing to
  Gemma-2-2b. The single-shot local-rotation numbers from Section 3 above agree exactly: Qwen
  0.4650 -> 0.0114, Gemma 0.4655 -> 0.4655 (unchanged to four decimal places).
* **Llama's non-monotonicity.** Llama does not degrade smoothly and does not stay flat: 0.4292 ->
  0.0114 (theta=0.10) -> 0.0364 (0.25) -> 0.1494 (0.50) -> 0.2645 (0.75) -> 0.2522 (1.00) -- a cliff
  at a small rotation angle, then a partial climb back as the rotation grows, leveling off well
  short of full recovery.

Fig. 1 puts all three curves on one axis and separates "how much did it drop" (panel B) from "what
does the curve's *shape* look like" (panel A), since these are really two different questions.


In [ ]:
# Fig. 1 -- rotation-angle sweep, all three models (numbers transcribed from the
# PART 4 cell above; this cell only re-plots and annotates them).
import numpy as np
import matplotlib.pyplot as plt

COLORS = {'Qwen2.5-1.5B': '#4C72B0', 'Gemma-2-2b': '#DD8452', 'Llama-3.2-1B': '#55A868'}
thetas = [0.00, 0.10, 0.25, 0.50, 0.75, 1.00]
rotation_sweep = {
    'Qwen2.5-1.5B': [0.4650, 0.0000, 0.0000, 0.0114, 0.0114, 0.0114],
    'Gemma-2-2b':   [0.4655, 0.4028, 0.4380, 0.4644, 0.4042, 0.4655],
    'Llama-3.2-1B': [0.4292, 0.0114, 0.0364, 0.1494, 0.2645, 0.2522],
}
baseline_acc  = {'Qwen2.5-1.5B': 0.4650, 'Gemma-2-2b': 0.4655, 'Llama-3.2-1B': 0.4292}
local_rot_acc = {'Qwen2.5-1.5B': 0.0114, 'Gemma-2-2b': 0.4655, 'Llama-3.2-1B': 0.2647}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axes[0]
for name, ys in rotation_sweep.items():
    lw = 3.2 if name == 'Llama-3.2-1B' else 2.0
    z  = 5 if name == 'Llama-3.2-1B' else 3
    ax.plot(thetas, ys, marker='o', label=name, color=COLORS[name], linewidth=lw, zorder=z)
llama = rotation_sweep['Llama-3.2-1B']
dip_idx = int(np.argmin(llama))
ax.annotate('dip', xy=(thetas[dip_idx], llama[dip_idx]),
            xytext=(thetas[dip_idx]-0.02, llama[dip_idx]-0.08), fontsize=9, color=COLORS['Llama-3.2-1B'])
ax.annotate('partial\nrecovery', xy=(thetas[-1], llama[-1]), xytext=(thetas[-1]-0.30, llama[-1]+0.08),
            fontsize=9, color=COLORS['Llama-3.2-1B'],
            arrowprops=dict(arrowstyle='->', color=COLORS['Llama-3.2-1B'], lw=1))
ax.set_xlabel(r'local MLP-axis rotation angle $\theta$ (0=identity, 1=full random rotation)')
ax.set_ylabel('next-token accuracy')
ax.set_title('Fig. 1  Rotation sweep: Qwen collapses and stays collapsed,\n'
             'Gemma is largely invariant, Llama dips then partially\n'
             'recovers (non-monotonic)')
ax.legend(fontsize=9, loc='center right'); ax.grid(alpha=0.25)

ax2 = axes[1]
models_ = list(rotation_sweep.keys())
rel_drop = [(baseline_acc[m]-local_rot_acc[m])/baseline_acc[m] for m in models_]
bars = ax2.bar(models_, rel_drop, color=[COLORS[m] for m in models_])
for b, v in zip(bars, rel_drop):
    ax2.text(b.get_x()+b.get_width()/2, v+0.02, f"{v:+.1%}", ha='center', fontsize=10)
ax2.set_ylabel('relative accuracy drop\nunder single-shot local rotation ($\\theta$=1)')
ax2.set_title('Fig. 1B  Relative drop under full local rotation')
ax2.set_ylim(-0.1, 1.1); ax2.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig('fig1_rotation_sweep.png', dpi=130)
plt.show()


**Reading Fig. 1.** Llama's curve is drawn thicker and annotated at its dip (theta=0.10) and its
partial recovery (theta=0.75-1.00) so the non-monotonicity is visible at a glance. Panel B turns
the Qwen/Gemma contrast into a single readable number per model: Qwen loses 97.5% of baseline
accuracy under a full local rotation, Gemma loses 0.0%, Llama sits in between at 38.3%.


### Point 3 -- architectural hypothesis for the Qwen/Gemma discrepancy

`PART 3, STEP B` (cross-family architecture differences, printed above) recorded the relevant
configuration facts:

| | Qwen2.5-1.5B | Gemma-2-2b | Llama-3.2-1B |
|---|---|---|---|
| activation | SiLU (SwiGLU) | GELU-tanh (GeGLU) | SiLU (SwiGLU) |
| GQA ratio | 12:2 | 8:4 | 32:8 |
| sliding window | none | 4096 (half the layers) | none |
| normalization | pre-norm only (RMSNorm) | pre-norm **and** post-norm on every attention/MLP
  sub-block | pre-norm only (RMSNorm) |

The normalization row is documented, not a guess: Gemma 2's technical report states RMSNorm is
applied to normalize both the input *and* output of every attention and feedforward sub-layer
(Team et al., 2024), on top of the family's larger input-embedding scale (sqrt(hidden_size)).
Two candidate mechanisms follow, and they make different predictions:

1. **"Post-norm directly repairs the rotation."** Not mechanistically plausible on its own: the
   rotation tested here is orthogonal, hence exactly norm-preserving, so renormalizing a
   sub-layer's output by its RMS cannot change how similar the rotated and unrotated versions of
   that same output are -- both already have the same norm before any renormalization.
2. **"Post-norm plus a larger residual stream dilutes the branch's relative contribution."** A
   rotation that completely scrambles one MLP branch's direction still only adds a small,
   bounded-magnitude vector to a much larger residual stream, so the *stream's* overall direction
   moves comparatively little.

**Real Check #1 and Real Check #2 below test these two mechanisms directly on the actual
checkpoints** -- hooking real submodules and real activations for all three model families,
rather than a synthetic replica built from random Gaussian weights. (An earlier version of this
notebook ran this section as two random-weight toy models; replaced here because a toy model
with the "right" dimensions can support a hypothesis without ever showing whether it holds on the
weights the hypothesis is actually about, or whether it generalizes across architectures instead
of being specific to whichever one motivated it.) Both are marked PENDING GPU EXECUTION for the
same reason as the rest of this notebook's real-checkpoint cells: no Hugging Face network access
in this authoring environment.


In [ ]:
# Real Check #1: does GEMMA-2's REAL post_feedforward_layernorm submodule causally reduce
# rotation sensitivity, tested by actually bypassing it on the real checkpoint -- not by
# renormalizing a random-weight toy MLP output. Qwen/Llama have no equivalent submodule, so
# their "native" curve doubles as the no-post-norm condition for a real, apples-to-apples,
# cross-family comparison (rather than synthetically inserting a module that architecture
# never had).
import torch.nn.functional as F
from contextlib import contextmanager

@contextmanager
def bypass_module(module):
    """Temporarily replace `module.forward` with the identity function, then restore the
    original -- a real causal ablation of a specific real submodule's real, learned weights,
    not a synthetic stand-in built to mimic what it might do."""
    original_forward = module.forward
    module.forward = lambda x: x
    try:
        yield module
    finally:
        module.forward = original_forward


def mlp_contribution_cos_batched(model, tokenizer, texts, layer_idx, device=DEVICE):
    """Cosine similarity, at real (non-pad) token positions, between the decoder layer's
    ACTUAL output (i.e. exactly what continues into the next layer, post any real post-norm
    submodule) under the model's own unrotated MLP weights vs. a rotated variant, at each
    theta -- one batched forward pass per theta/condition on the real checkpoint."""
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    enc = tokenizer(texts, return_tensors='pt', padding=True).to(device)
    mask = enc.attention_mask.bool()
    layer = get_layers(model)[layer_idx]

    def capture_layer_output(m):
        captured = {}
        def hook(mod, inp, out):
            captured['out'] = (out[0] if isinstance(out, tuple) else out).detach()
        h = layer.register_forward_hook(hook)
        with torch.inference_mode():
            m(enc.input_ids, attention_mask=enc.attention_mask)
        h.remove()
        return captured['out'][mask].float()

    base_out = capture_layer_output(model)
    results = {}
    for theta, cond in rotation_angle_sweep_mlp(model, layer_idx, [0.10, 0.25, 0.50, 0.75, 1.00],
                                                 seed=0, device=device):
        rot_out = capture_layer_output(cond)
        results[theta] = F.cosine_similarity(base_out, rot_out, dim=-1).mean().item()
        del rot_out
    return results


print("=" * 78)
print("REAL CHECK #1: DOES A REAL POST-NORM SUBMODULE CAUSALLY REDUCE ROTATION SENSITIVITY?")
print("=" * 78)

gemma_layer = get_layers(models['Gemma-2-2b'])[TARGET_LAYER]
has_post_norm = hasattr(gemma_layer, 'post_feedforward_layernorm')
print(f"Gemma-2-2b layer {TARGET_LAYER} has post_feedforward_layernorm: {has_post_norm}")
print("(If this reads False, the installed transformers version's Gemma2DecoderLayer doesn't")
print(" expose the sandwich-norm submodule under this attribute name -- check `dir(gemma_layer)`")
print(" before trusting anything below; the bypass step is skipped automatically either way.)")

real_check1_results = {}

print("\n-- Gemma-2-2b, post-norm ON (native) --")
real_check1_results['Gemma-2-2b (native, post-norm ON)'] = mlp_contribution_cos_batched(
    models['Gemma-2-2b'], tokenizers['Gemma-2-2b'], EVAL_CORPUS, TARGET_LAYER)
for th, cs in real_check1_results['Gemma-2-2b (native, post-norm ON)'].items():
    print(f"  theta={th:.2f}: cos.sim={cs:.4f}")

if has_post_norm:
    print("\n-- Gemma-2-2b, post-norm BYPASSED (real submodule, real weights, forward replaced "
          "with identity) --")
    with bypass_module(gemma_layer.post_feedforward_layernorm):
        real_check1_results['Gemma-2-2b (post-norm BYPASSED)'] = mlp_contribution_cos_batched(
            models['Gemma-2-2b'], tokenizers['Gemma-2-2b'], EVAL_CORPUS, TARGET_LAYER)
    for th, cs in real_check1_results['Gemma-2-2b (post-norm BYPASSED)'].items():
        print(f"  theta={th:.2f}: cos.sim={cs:.4f}")

for name in ['Qwen2.5-1.5B', 'Llama-3.2-1B']:
    print(f"\n-- {name} (native -- no post-feedforward-norm submodule in this architecture) --")
    real_check1_results[f'{name} (native, no post-norm)'] = mlp_contribution_cos_batched(
        models[name], tokenizers[name], EVAL_CORPUS, TARGET_LAYER)
    for th, cs in real_check1_results[f'{name} (native, no post-norm)'].items():
        print(f"  theta={th:.2f}: cos.sim={cs:.4f}")
    free_memory()

fig, ax = plt.subplots(figsize=(7.5, 5))
for label, results in real_check1_results.items():
    thetas_ = list(results.keys()); vals_ = list(results.values())
    style = '--' if 'BYPASSED' in label else '-'
    marker = 'x' if 'BYPASSED' in label else 'o'
    ax.plot(thetas_, vals_, marker=marker, linestyle=style, label=label)
ax.set_xlabel('rotation angle theta')
ax.set_ylabel("cos.sim of the decoder layer's actual output")
ax.set_title("Real Check #1  Bypassing Gemma-2's real post-norm --\n"
             "does it move toward Qwen/Llama's native curves?")
ax.legend(fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout(); plt.savefig('real_check1_postnorm_ablation.png', dpi=130); plt.show()

print("\nPrediction if post-norm is doing real causal work: the BYPASSED curve sits BELOW the")
print("native (post-norm ON) curve at every theta -- more sensitive to rotation once the real")
print("mechanism is removed -- and shifts toward the Qwen/Llama native curves specifically, not")
print("just away from Gemma's own native curve in some arbitrary direction. If the BYPASSED curve")
print("barely moves, post-norm alone doesn't explain Gemma's invariance and Real Check #2 below")
print("(the dilution ratio, independent of any specific normalization submodule) is doing more of")
print("the explanatory work.")


**Reading Real Check #1.** This is a real causal intervention on Gemma-2-2b's actual, trained
`post_feedforward_layernorm` weights, not a description of what post-norm "should" do in theory --
so the direction and size of the BYPASSED-vs-native gap is an empirical result to read off the
printed numbers and Fig. above, not assumed here. What makes this check more informative than the
earlier synthetic version: (1) it uses the model's own real MLP weights at every theta, so any
gap is guaranteed to be about *this* architecture's actual learned parameters, and (2) running the
same probe on Qwen and Llama (which have no such module at all) gives a real second data point for
"no post-norm" beyond just "post-norm bypassed on Gemma" -- if Gemma-bypassed lands close to
Qwen/Llama-native, that is much stronger evidence for the post-norm mechanism than a within-Gemma
comparison alone could be.


In [ ]:
# Real Check #2: does the branch/residual DILUTION RATIO actually predict rotation-invariance
# on real weights, and does the SAME relationship hold across all three architectures, or is it
# model-specific? Every number below (the ratio and the cosine similarity) is read off a real
# forward pass through the real checkpoint at each theta -- no random Gaussian weights anywhere.
# Supersedes the baseline-only branch/residual ratio check further below (Point 3 follow-up A),
# which this extends into a full theta sweep correlated against actual downstream disruption.
import torch.nn.functional as F

def dilution_probe_batched(model, tokenizer, texts, layer_idx, device=DEVICE):
    """Real, per-theta measurement of (a) mean ||MLP branch output|| / ||residual stream
    entering that layer|| and (b) cosine similarity of the FULL decoder-layer output (after
    any real post-norm, attention, GQA structure, etc.) between the rotated and baseline
    conditions -- the real-checkpoint analogue of the synthetic toy check #2 sweep, with both
    quantities read off the actual model at every theta rather than fixed by hand."""
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    enc = tokenizer(texts, return_tensors='pt', padding=True).to(device)
    mask = enc.attention_mask.bool()
    layer = get_layers(model)[layer_idx]
    mlp = get_mlp(model, layer_idx)

    def run(m):
        captured = {}
        def hook_pre(mod, inp):
            captured['residual_in'] = inp[0].detach()
        def hook_out(mod, inp, out):
            captured['layer_out'] = (out[0] if isinstance(out, tuple) else out).detach()
        def hook_mlp(mod, inp, out):
            captured['mlp_out'] = out.detach()
        h1 = layer.register_forward_pre_hook(hook_pre)
        h2 = layer.register_forward_hook(hook_out)
        h3 = mlp.register_forward_hook(hook_mlp)
        with torch.inference_mode():
            m(enc.input_ids, attention_mask=enc.attention_mask)
        h1.remove(); h2.remove(); h3.remove()
        return captured

    base = run(model)
    r = base['residual_in'].float().norm(dim=-1)[mask]
    base_layer_out = base['layer_out'][mask].float()

    results = {}
    for theta, cond in rotation_angle_sweep_mlp(model, layer_idx, [0.10, 0.25, 0.50, 0.75, 1.00],
                                                 seed=0, device=device):
        rot = run(cond)
        m_norm = rot['mlp_out'].float().norm(dim=-1)[mask]
        ratio = float((m_norm / (r + 1e-8)).mean().item())
        cos = F.cosine_similarity(base_layer_out, rot['layer_out'][mask].float(), dim=-1).mean().item()
        results[theta] = dict(ratio=ratio, cos=cos)
        del rot
    return results


print("=" * 78)
print("REAL CHECK #2: DOES THE DILUTION RATIO PREDICT ROTATION-INVARIANCE ON REAL WEIGHTS?")
print("=" * 78)

real_check2_results = {}
for name in MODEL_CONFIGS:
    print(f"\n-- {name} --")
    res = dilution_probe_batched(models[name], tokenizers[name], EVAL_CORPUS, TARGET_LAYER)
    real_check2_results[name] = res
    for theta, d in res.items():
        print(f"  theta={theta:.2f}: branch/residual ratio={d['ratio']:.4f}   "
              f"layer-output cos.sim={d['cos']:.4f}")
    free_memory()

fig, ax = plt.subplots(figsize=(7.5, 5))
for name in MODEL_CONFIGS:
    res = real_check2_results[name]
    ratios = [d['ratio'] for d in res.values()]
    coss = [d['cos'] for d in res.values()]
    ax.plot(ratios, coss, marker='o', label=name, color=COLORS[name])
ax.set_xlabel("mean ||MLP branch output|| / ||residual stream|| (under that theta's rotation)")
ax.set_ylabel('cos.sim of the full decoder-layer output vs. theta=0 baseline')
ax.set_title('Real Check #2  Same dilution-predicts-invariance relationship\n'
             'across three real architectures, or model-specific?')
ax.legend(fontsize=9); ax.grid(alpha=0.25)
plt.tight_layout(); plt.savefig('real_check2_dilution_cross_model.png', dpi=130); plt.show()

print("\nIf the mechanism generalizes, all three models' points should fall on roughly the SAME")
print("downward-sloping ratio-vs-cos.sim curve (higher relative branch magnitude -> more")
print("disruption), even though each model reaches different (ratio, theta) pairs -- that would")
print("be evidence the relationship is a property of the dilution ratio itself, not of any one")
print("architecture. If instead the three models trace clearly SEPARATE curves at the SAME ratio")
print("value, dilution alone doesn't explain the cross-model gap, and something architecture-")
print("specific (e.g. Gemma's post-norm, tested causally in Real Check #1 above) is doing")
print("additional explanatory work beyond what the ratio alone captures.")


**Reading Real Check #2.** The key comparison is whether the three models' curves in the figure
above are one shared curve or three separate ones. A shared curve is the stronger, more useful
finding: it would mean the dilution ratio is doing real explanatory work as a general property of
transformer MLP blocks, not a coincidence specific to whichever model motivated the hypothesis --
directly the kind of cross-model generalization check the toy-model version could not do, since a
single synthetic replica has no second architecture to compare itself against. Separate curves at
matching ratios would be an equally real and equally useful finding: it would mean ratio alone is
an incomplete explanation and something else (post-norm placement, activation function, GQA
structure) is contributing on top of it -- exactly what Real Check #1's causal post-norm ablation,
and Further experiment A's head-count/RoPE sweep below, are designed to isolate.


### Further experiment A -- ruling in/out attention-head count and RoPE (Point 3)

The reviewer named two other candidate mechanisms: **rotational invariance from positional
embeddings**, and **inductive biases in the attention heads**. Testable the same way: if more
attention heads give a downstream layer more independent subspaces to average out a locally
corrupted input direction, a downstream attention layer's output should diverge *less* from
baseline after the same corrupting rotation as head count increases -- using each model's *real*
query-head count (Qwen 12, Gemma 8, Llama 32) makes this a direct, checkable prediction rather
than an arbitrary sweep. Toggling RoPE on/off in the same toy layer isolates the positional-
embedding variable separately. If neither moves the needle, that's informative too: it means the
dilution story from Real Check #2 above (now measured on the real checkpoints, not a synthetic
replica) isn't competing against an equally-plausible untested alternative. This cell still uses a
small synthetic attention layer rather than the real checkpoints, since isolating head-count and
RoPE as independent variables (all combinations, controlled, on the same underlying activations)
isn't something you can ask three fixed real architectures to give you jointly -- each real model
only gives one head-count/RoPE combination, not a full factorial sweep.

**HPC note:** all 3 head-configs x 2 RoPE-settings x 20 trials below run as small, fully
batched tensor ops (no Python-level per-token loop) -- the whole cell is a few hundred ms even on
CPU, so no GPU is needed for this synthetic check at all.


In [ ]:
# Further experiment A: single downstream causal multi-head attention layer, fed the
# SAME baseline-vs-rotated residual stream from toy check #2, varying (i) number of
# query heads (real head counts: Qwen=12, Gemma=8, Llama=32) and (ii) RoPE on/off.
import torch, torch.nn.functional as F
import numpy as np
from scipy.linalg import expm

torch.manual_seed(0)
d_model, d_ff, T = 384, 1536, 16   # d_model divisible by 12, 8, and 32

def rope_matrices(T, d_head, base=10000.0):
    half = d_head // 2
    freqs = 1.0 / (base ** (torch.arange(0, half).float() / half))
    angles = torch.outer(torch.arange(T).float(), freqs)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin):
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

def multihead_attn(resid, n_heads, use_rope, Wq, Wk, Wv, Wo):
    d_head = d_model // n_heads
    Q = (resid @ Wq).view(T, n_heads, d_head)
    K = (resid @ Wk).view(T, n_heads, d_head)
    V = (resid @ Wv).view(T, n_heads, d_head)
    if use_rope:
        cos, sin = rope_matrices(T, d_head)
        Q = torch.stack([apply_rope(Q[:, h], cos, sin) for h in range(n_heads)], dim=1)
        K = torch.stack([apply_rope(K[:, h], cos, sin) for h in range(n_heads)], dim=1)
    scores = torch.einsum('thd,shd->ths', Q, K) / (d_head ** 0.5)
    mask = torch.tril(torch.ones(T, T)).bool()
    scores = scores.masked_fill(~mask.unsqueeze(1), float('-inf'))
    attn = scores.softmax(dim=-1)
    out = torch.einsum('ths,shd->thd', attn, V).reshape(T, n_heads * d_head)
    return out @ Wo

head_counts = {'Qwen-like (12 heads)': 12, 'Gemma-like (8 heads)': 8, 'Llama-like (32 heads)': 32}
n_trials = 20
results = {}
with torch.inference_mode():
    for label, n_heads in head_counts.items():
        for use_rope in [False, True]:
            css = []
            for trial in range(n_trials):
                torch.manual_seed(100 + trial)
                Wq = torch.randn(d_model, d_model) / d_model**0.5
                Wk = torch.randn(d_model, d_model) / d_model**0.5
                Wv = torch.randn(d_model, d_model) / d_model**0.5
                Wo = torch.randn(d_model, d_model) / d_model**0.5
                W_gate = torch.randn(d_ff, d_model) / d_model**0.5
                W_up   = torch.randn(d_ff, d_model) / d_model**0.5
                W_down = torch.randn(d_model, d_ff) / d_ff**0.5

                resid = torch.randn(T, d_model)
                H = F.silu(resid @ W_gate.T) * (resid @ W_up.T)
                total_base = resid + (H @ W_down.T)

                B = torch.randn(d_ff, d_ff); A = (B - B.T) / 2
                Q_full = torch.tensor(expm(A.numpy().astype(np.float64)), dtype=torch.float32)
                total_rot = resid + ((H @ Q_full.T) @ W_down.T)

                out_base = multihead_attn(total_base, n_heads, use_rope, Wq, Wk, Wv, Wo)
                out_rot  = multihead_attn(total_rot,  n_heads, use_rope, Wq, Wk, Wv, Wo)
                css.append(F.cosine_similarity(out_base, out_rot, dim=-1).mean().item())
            results[(label, use_rope)] = (np.mean(css), np.std(css))

print(f"{'config':>25} | {'no RoPE (mean +/- sd)':>22} | {'with RoPE (mean +/- sd)':>23}")
for label in head_counts:
    a_m, a_s = results[(label, False)]
    b_m, b_s = results[(label, True)]
    print(f"{label:>25} | {a_m:8.4f} +/- {a_s:.4f}   | {b_m:8.4f} +/- {b_s:.4f}")


**Evidence from experiment A.** All six conditions land in a tight band (0.51-0.52), well
within one standard deviation of each other -- neither head count (spanning the real range across
all three families) nor RoPE moves a downstream attention layer's sensitivity to an upstream
corrupted direction here, while toy check #2's dilution ratio produced a clean, monotonic,
order-of-magnitude effect (0.13 to 0.9996). **Both of the reviewer's other two candidate
mechanisms fail to differentiate the models here**, which raises confidence that the
dilution/normalization story is doing real explanatory work rather than being one hypothesis
among several equally-supported ones. Still a toy result, not a measurement on the real
checkpoints -- see `branch_to_residual_norm_ratio` in the PENDING-GPU cell below for the direct
test.


### Real-model follow-up for Point 3 (PENDING GPU execution -- not run here)

This authoring environment has no network access to `huggingface.co`, so this cell is written to
the same `ConditionModel` / `MODEL_CONFIGS` conventions used throughout this notebook and is ready
to run in the same Colab/Kaggle session, but has not been executed. Its branch/residual-ratio
probe (previously here) has been superseded by Real Check #2 above, which measures the same
quantity across the full rotation sweep rather than only at baseline -- this cell now covers only
the non-linguistic digit-copy/digit-add follow-up (Point 3 Follow-up B). **HPC note:** both probes
below batch every text in `EVAL_CORPUS` (or the simple-task example set) into a single padded
forward pass per model per condition, instead of the one-sequence-at-a-time Python loop used by
`next_token_accuracy` elsewhere in this notebook -- on a T4 that's the difference between ~6 launches and dozens for the
simple-task sweep across 6 theta values x 2 tasks x 3 models. Padding is right-padded with
`attention_mask` supplied to `model(...)`, so causal attention still ignores pad positions
correctly for every real token's prediction.


**Also added this revision:** a shared `_amp_ctx()` autocast helper (matches the notebook-wide `DTYPE`, so a T4 gets fp16 tensor-core matmuls and an Ampere+ box gets bf16, consistent with how the checkpoints were loaded -- see the DTYPE fix in Part 0), plus `free_memory()` / `print_gpu_mem()` / per-model wall-clock timing around each model's iteration in both loops below, since three ~1-3B models sharing one T4 means peak memory during the largest of these batched forward passes is worth watching, not just the fact that it fits at all.

In [ ]:
# PENDING GPU EXECUTION -- not run in this environment (no Hugging Face network access here).
# Run this in the same session as the rest of this notebook, after MODEL_CONFIGS/models/
# tokenizers exist.

import time
from contextlib import nullcontext

def _amp_ctx():
    # Matches the checkpoints' own load dtype (see the DTYPE fix in Part 0): fp16 autocast
    # on Turing (T4/P100, real tensor-core speedup), bf16 on Ampere+, a no-op context on CPU.
    if DEVICE == 'cuda':
        return torch.autocast(device_type='cuda', dtype=DTYPE)
    return nullcontext()


# POINT 3 FOLLOW-UP A (baseline-only branch/residual ratio) is now SUPERSEDED by Real Check #2
# above, which measures this same ratio at every theta in the rotation sweep -- not just at
# baseline -- and correlates it against actual downstream cosine similarity, across all three
# real checkpoints. Left out here to avoid computing the same quantity twice; see Real Check #2's
# printed output and Fig. "real_check2_dilution_cross_model.png" for this data.

# --- simple non-linguistic probes: digit copy and single-digit addition, also batched ---
DIGIT_COPY = [(f"{d} {d} {d} -> ", f" {d}") for d in "0123456789"]
DIGIT_ADD  = [(f"{a}+{b}=", f" {a+b}") for a in range(1, 5) for b in range(1, 5)]

def simple_task_accuracy_batched(model, tokenizer, examples, device=DEVICE):
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    prompts, targets = zip(*examples)
    enc = tokenizer(list(prompts), return_tensors='pt', padding=True).to(device)
    target_ids = torch.tensor(
        [tokenizer(t, add_special_tokens=False).input_ids[0] for t in targets], device=device)
    with torch.inference_mode(), _amp_ctx():
        logits = model(enc.input_ids, attention_mask=enc.attention_mask).logits
    # last REAL (non-pad) position per row, since prompts have different lengths
    last_real = enc.attention_mask.sum(dim=1) - 1
    pred_ids = logits[torch.arange(logits.shape[0]), last_real].argmax(dim=-1)
    return (pred_ids == target_ids).float().mean().item()

print()
print("=" * 78)
print("POINT 3 FOLLOW-UP B: ROTATION ON SIMPLE NON-LINGUISTIC TASKS (DIGIT COPY / ADD)")
print("=" * 78)
SIMPLE_THETAS = [0.00, 0.10, 0.25, 0.50, 0.75, 1.00]
for name in MODEL_CONFIGS:
    t0 = time.perf_counter()
    model, tok = models[name], tokenizers[name]
    base_copy = simple_task_accuracy_batched(model, tok, DIGIT_COPY)
    base_add  = simple_task_accuracy_batched(model, tok, DIGIT_ADD)
    print(f"  {name:>15}: baseline digit-copy={base_copy:.3f}  digit-add={base_add:.3f}")
    for theta, cond in rotation_angle_sweep_mlp(model, TARGET_LAYER, SIMPLE_THETAS, seed=0, device=DEVICE):
        copy_acc = simple_task_accuracy_batched(cond, tok, DIGIT_COPY)
        add_acc  = simple_task_accuracy_batched(cond, tok, DIGIT_ADD)
        print(f"      theta={theta:.2f}: digit-copy={copy_acc:.3f}  digit-add={add_acc:.3f}")
    print(f"  ({name} follow-up B total: {time.perf_counter()-t0:.1f}s)")
    print_gpu_mem(f"after {name} follow-up B")
    free_memory()
print("\nIf the Qwen/Gemma split is a general property of local rotation (not specific to")
print("natural-language next-token prediction), it should reproduce here.")


### Point 4 -- Llama's non-monotonic degradation

Llama-3.2-1B's curve (0.4292 -> 0.0114 -> 0.0364 -> 0.1494 -> 0.2645 -> 0.2522) falls fastest of
the three at the smallest rotation tested, then climbs back to ~60% of baseline by theta=0.75-1.00
without ever fully recovering. Two structurally different explanations imply different follow-ups:

1. **Redundant/degenerate circuits.** If several subsets of Llama's 8192-wide MLP hidden layer
   carry overlapping, partially redundant information ("superposition," Elhage et al., 2022), a
   large rotation could land closer to a *different* redundant subset than a small one does,
   producing non-monotonicity without any single "special" angle.
2. **A near-return in a low-effective-rank read-out.** If `W_down` is effectively low rank, a large
   rotation might partially re-align with *some* direction the readout still responds to, even if
   not the same direction it started from.


### Further experiment B -- redundant circuits, or generic rotation geometry? (Point 4)

The reviewer's proposal names two specific mechanisms: **redundant circuits** and **temporary
symmetry orbits**. Both make checkable, different predictions: if redundant circuits are
responsible, a `W_down` built with explicit near-duplicate neuron groups should show *more*
pronounced non-monotonicity than an equivalent full-rank random `W_down`. If a **temporary
symmetry orbit** is responsible, then at the theta where output similarity recovers, the *rotated*
hidden vector should be well-approximated by *some permutation* of the *original* hidden vector's
neurons (found via the Hungarian algorithm, the same exact-permutation-recovery machinery used
throughout this notebook's Part 2), even though it isn't close to the original directly.


In [ ]:
# Further experiment B: redundant (superposed) vs. full-rank W_down, continuous rotation
# sweep, comparing (i) raw output cosine similarity and (ii) best-permutation-aligned
# neuron similarity (Hungarian algorithm).
import torch, torch.nn.functional as F
import numpy as np
from scipy.linalg import expm
from scipy.optimize import linear_sum_assignment

torch.manual_seed(3)
d_model, d_ff, N = 64, 96, 200
group_size = 4  # each "role" duplicated group_size times -> redundant/superposed neurons

def redundant_down_proj():
    n_roles = d_ff // group_size
    base_cols = torch.randn(d_model, n_roles) / d_model**0.5
    W = base_cols.repeat_interleave(group_size, dim=1) / group_size**0.5
    W += 0.05 * torch.randn(d_model, d_ff) / d_ff**0.5
    return W

def full_rank_down_proj():
    return torch.randn(d_model, d_ff) / d_ff**0.5

W_gate = torch.randn(d_ff, d_model) / d_model**0.5
W_up   = torch.randn(d_ff, d_model) / d_model**0.5
X = torch.randn(N, d_model)
H = F.silu(X @ W_gate.T) * (X @ W_up.T)

B = torch.randn(d_ff, d_ff); A = (B - B.T) / 2.0
A_np = A.numpy().astype(np.float64)
thetas = np.linspace(0, 1.2, 13)

def permutation_aligned_cos(h_base, h_rot):
    hb, hr = h_base.numpy(), h_rot.numpy()
    hb_n = hb / (np.linalg.norm(hb, axis=0, keepdims=True) + 1e-8)
    hr_n = hr / (np.linalg.norm(hr, axis=0, keepdims=True) + 1e-8)
    sim = hb_n.T @ hr_n / hb.shape[0]
    row_ind, col_ind = linear_sum_assignment(-sim)
    return sim[row_ind, col_ind].mean()

for label, Wdown_fn in [("redundant (superposed) W_down", redundant_down_proj),
                         ("full-rank random W_down", full_rank_down_proj)]:
    W_down = Wdown_fn()
    out_base = H @ W_down.T
    print(f"\n-- {label} --")
    print(f"{'theta':>6} | {'raw output cos.sim':>19} | {'best-permutation neuron-cos.sim':>32}")
    for theta in thetas:
        Q = torch.tensor(expm(theta * A_np), dtype=torch.float32)
        H_rot = H @ Q.T
        out_rot = H_rot @ W_down.T
        raw_cs = F.cosine_similarity(out_base, out_rot, dim=-1).mean().item()
        perm_cs = permutation_aligned_cos(H, H_rot)
        print(f"{theta:6.2f} | {raw_cs:19.4f} | {perm_cs:32.4f}")


**Evidence from experiment B, and it complicates rather than confirms the reviewer's proposed
explanation.** Both `W_down` conditions show the *same* qualitative shape: a fall to negative
cosine around theta=0.3-0.4, then a partial climb to ~0.05-0.09 by theta=1.0-1.1. The redundant
and full-rank versions are nearly indistinguishable (0.0926 vs. 0.0548 at theta=1.00, within the
run-to-run variation seen elsewhere in this notebook) -- **evidence against redundant circuits
being *necessary*** for this shape: a dip-then-partial-recovery pattern looks like a fairly
generic consequence of rotating a nonlinear (SwiGLU) hidden layer through a *linear*
down-projection, independent of whether that projection has redundant structure. The
permutation-aligned neuron similarity stays pinned near zero throughout in both conditions --
**no theta shows an accidental permutation-orbit recovery.** Combined with experiment A: two of
the reviewer's three candidate mechanisms (head count, RoPE, and now the specific
permutation-orbit account) show no effect in controlled toy checks, while the
dilution/normalization mechanism and a generic (non-redundancy-specific) geometric explanation for
Llama both produced clean, reproducible signal. Whether the generic-geometry account or something
specific to Llama's *trained* weights (which this random-Gaussian toy model cannot capture) is the
right account for the real checkpoint is exactly what the PENDING-GPU cell below is designed to
test.


### Further experiment C -- a low-effective-rank readout, the reviewer's second Point-4 mechanism

Point 4 named two candidate mechanisms up front: **redundant circuits** (tested in experiment B
above -- ruled out as *necessary*, since a full-rank random `W_down` reproduced the same
dip-then-partial-recovery shape as a deliberately redundant one) and **a near-return in a
low-effective-rank readout**, which experiment B did not actually test. Those are genuinely
different claims: "redundant" means specific columns of `W_down` are near-duplicates of each
other; "low effective rank" means the column *space* is low-dimensional, which a matrix with no
duplicated columns at all can still have (e.g. columns drawn from a random low-dimensional
subspace with no two of them close to parallel). Experiment B's redundant construction happens to
also be low-rank, so it never isolated rank from duplication as separate variables -- this cell
does, building `W_down` at several *controlled numerical ranks* via truncated-SVD reconstruction
(orthogonal factors, no repeated or near-parallel columns at any rank), and reports each
construction's actual effective rank (participation ratio of its singular-value spectrum) rather
than just its constructor's intended rank, since numerical rank and intended rank aren't always
identical once floating-point noise is added back in.

**Prediction if rank (not redundancy) drives the recovery shape:** the dip depth and/or the
theta=1.0 recovery height should trend systematically with rank, not sit flat across rank the way
experiment B's redundant-vs-full-rank comparison did.


In [ ]:
# Further experiment C: W_down built at controlled numerical rank (via truncated SVD --
# orthogonal factors, no duplicated or near-parallel columns at any rank, isolating rank from
# redundancy), continuous rotation sweep, raw output cosine similarity + permutation-aligned
# neuron similarity (same Hungarian-algorithm probe as experiment B) at each rank.
import torch, torch.nn.functional as F
import numpy as np
from scipy.linalg import expm
from scipy.optimize import linear_sum_assignment

torch.manual_seed(4)
d_model, d_ff, N = 64, 96, 200

def rank_controlled_down_proj(rank, gen):
    # Build a d_model x d_ff matrix with EXACT numerical rank `rank`: random orthonormal
    # factors U (d_model x rank) and V (d_ff x rank), random positive singular values, no
    # duplicated or near-parallel columns at any rank (unlike the repeat_interleave
    # construction in experiment B, which is low-rank BY duplication).
    U, _ = torch.linalg.qr(torch.randn(d_model, d_model, generator=gen))
    V, _ = torch.linalg.qr(torch.randn(d_ff, d_ff, generator=gen))
    U, V = U[:, :rank], V[:, :rank]
    s = torch.rand(rank, generator=gen) + 0.5              # singular values in [0.5, 1.5)
    W = (U * s) @ V.T
    return W / W.norm() * (d_model * d_ff) ** 0.5 / 10.0    # match experiment B's rough scale

def effective_rank(W):
    # Participation ratio of the singular-value spectrum: (sum s)^2 / sum(s^2). Equals the
    # true rank for a flat spectrum, and shrinks toward 1 as the spectrum concentrates on one
    # direction -- a continuous measure of "how low is low", not just the constructor's intent.
    s = torch.linalg.svdvals(W)
    return float((s.sum() ** 2 / (s ** 2).sum()).item())

W_gate = torch.randn(d_ff, d_model) / d_model**0.5
W_up   = torch.randn(d_ff, d_model) / d_model**0.5
X = torch.randn(N, d_model)
H = F.silu(X @ W_gate.T) * (X @ W_up.T)

B = torch.randn(d_ff, d_ff); A = (B - B.T) / 2.0
A_np = A.numpy().astype(np.float64)
thetas = np.linspace(0, 1.2, 13)

def permutation_aligned_cos(h_base, h_rot):
    hb, hr = h_base.numpy(), h_rot.numpy()
    hb_n = hb / (np.linalg.norm(hb, axis=0, keepdims=True) + 1e-8)
    hr_n = hr / (np.linalg.norm(hr, axis=0, keepdims=True) + 1e-8)
    sim = hb_n.T @ hr_n / hb.shape[0]
    row_ind, col_ind = linear_sum_assignment(-sim)
    return sim[row_ind, col_ind].mean()

ranks_to_test = [64, 32, 16, 8, 4]   # d_model=64 -> first entry is full rank
gen = torch.Generator().manual_seed(4)
dip_depths, recovery_heights, eff_ranks = [], [], []
curves = {}

for rank in ranks_to_test:
    W_down = rank_controlled_down_proj(rank, gen)
    eff_rank = effective_rank(W_down)
    out_base = H @ W_down.T
    raw_curve = []
    print(f"\n-- rank={rank} (effective rank={eff_rank:.2f}) --")
    print(f"{'theta':>6} | {'raw output cos.sim':>19} | {'best-permutation neuron-cos.sim':>32}")
    for theta in thetas:
        Q = torch.tensor(expm(theta * A_np), dtype=torch.float32)
        H_rot = H @ Q.T
        out_rot = H_rot @ W_down.T
        raw_cs = F.cosine_similarity(out_base, out_rot, dim=-1).mean().item()
        perm_cs = permutation_aligned_cos(H, H_rot)
        raw_curve.append(raw_cs)
        print(f"{theta:6.2f} | {raw_cs:19.4f} | {perm_cs:32.4f}")
    curves[rank] = raw_curve
    dip_depths.append(min(raw_curve))
    recovery_heights.append(raw_curve[-1])
    eff_ranks.append(eff_rank)

print("\n" + "=" * 78)
print(f"{'rank':>6} | {'effective rank':>14} | {'dip depth (min cos.sim)':>24} | {'theta=1.2 recovery':>19}")
for rank, er, dd, rh in zip(ranks_to_test, eff_ranks, dip_depths, recovery_heights):
    print(f"{rank:6d} | {er:14.2f} | {dd:24.4f} | {rh:19.4f}")

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
cmap = plt.cm.viridis(np.linspace(0.15, 0.9, len(ranks_to_test)))
for rank, color in zip(ranks_to_test, cmap):
    axes[0].plot(thetas, curves[rank], marker='o', markersize=3, color=color,
                 label=f"rank={rank} (eff.={eff_ranks[ranks_to_test.index(rank)]:.1f})")
axes[0].set_xlabel(r'rotation angle $\theta$'); axes[0].set_ylabel('raw output cos.sim')
axes[0].set_title('Fig. 3A  Rotation sweep by controlled numerical rank\n(no duplicated columns at any rank)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.25)

axes[1].plot(eff_ranks, dip_depths, marker='o', label='dip depth (min cos.sim)', color='#C44E52')
axes[1].plot(eff_ranks, recovery_heights, marker='s', label='theta=1.2 recovery', color='#4C72B0')
axes[1].set_xlabel('effective rank (participation ratio)')
axes[1].set_ylabel('cosine similarity')
axes[1].set_title('Fig. 3B  Does rank predict dip depth / recovery height?')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.25)
plt.tight_layout(); plt.savefig('fig3_rank_controlled.png', dpi=130); plt.show()


**Reading experiment C.** If Fig. 3B's two curves are roughly flat across effective rank, that
extends experiment B's conclusion: rank alone (independent of redundancy) also fails to
differentiate the dip-then-partial-recovery shape, strengthening the case that this is a generic
consequence of composing a nonlinear hidden layer with *any* linear down-projection under
rotation, not a signature of either redundant circuits or a low-rank readout specifically. If
instead dip depth or recovery height trend systematically with effective rank, that's a genuinely
new finding experiment B's redundancy-confounded construction could not have surfaced -- in that
case the honest next step is checking the REAL checkpoints' actual `down_proj` singular-value
spectra (cheap: one `torch.linalg.svdvals` call per layer, no forward pass needed) against this
toy model's rank-vs-shape relationship, which the PENDING-GPU cell below does not yet cover and
would be a natural next addition once this cell's own result is in.


### Real-model follow-up for Point 4 (PENDING GPU execution -- not run here)

Also not executed here for the same network-access reason. Reuses `rotation_angle_sweep_mlp` and
`linear_cka` exactly as already defined above, plus a new `attention_entropy` probe. **HPC note:**
`layerwise_cka_under_rotation` collects every layer's hidden states for the WHOLE `EVAL_CORPUS` in
one batched, padded forward pass per theta (6 forward passes total per model, not
`len(EVAL_CORPUS) x 6`), and `attention_entropy` likewise runs one batched forward with
`output_attentions=True` instead of a per-text loop -- on a 16-layer model over an 8-sentence
corpus that's the difference between ~6 forward passes and ~48+ for this cell alone.


In [ ]:
# PENDING GPU EXECUTION -- not run in this environment (no Hugging Face network access here).

import time
from contextlib import nullcontext

def _amp_ctx():
    if DEVICE == 'cuda':
        return torch.autocast(device_type='cuda', dtype=DTYPE)
    return nullcontext()


# BUG FIX (real Kaggle run surfaced this): these checkpoints load with the default
# `sdpa` attention implementation (faster, fused, no materialized attention-weight
# tensor), which does NOT support `output_attentions=True` -- it doesn't raise, it just
# sets `out.attentions = None`, so `out.attentions[layer_idx]` crashed with `TypeError:
# 'NoneType' object is not subscriptable` deep inside this function, far from the actual
# cause. `eager` attention materializes the real per-head attention-weight tensor this
# probe needs; it's slower and higher-memory than sdpa, so it's switched in ONLY for the
# duration of this specific call and restored immediately after, rather than reloading
# any model or leaving eager on for the rest of the notebook.
from contextlib import contextmanager

@contextmanager
def eager_attention(model):
    if hasattr(model, 'set_attn_implementation'):
        original = model.config._attn_implementation
        model.set_attn_implementation('eager')
        try:
            yield model
        finally:
            model.set_attn_implementation(original)
    else:
        # older transformers versions without the setter: attention modules read
        # `self.config._attn_implementation` directly at forward time, so mutating the
        # shared config object in place has the same effect.
        original = model.config._attn_implementation
        model.config._attn_implementation = 'eager'
        try:
            yield model
        finally:
            model.config._attn_implementation = original


def attention_entropy_batched(model, tokenizer, texts, layer_idx, device=DEVICE):
    """Mean entropy (nats) of the attention distribution at `layer_idx`, over real (non-pad)
    query positions, heads, and texts -- one batched forward pass, not a per-text loop."""
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    enc = tokenizer(list(texts), return_tensors='pt', padding=True).to(device)
    with torch.inference_mode(), _amp_ctx(), eager_attention(model):
        out = model(enc.input_ids, attention_mask=enc.attention_mask, output_attentions=True)
    if out.attentions is None or out.attentions[layer_idx] is None:
        raise RuntimeError(
            f"output_attentions still returned None at layer {layer_idx} even under eager "
            f"attention -- check model.config._attn_implementation actually reads back as "
            f"'eager' (print it), and that this transformers version's modeling code for "
            f"this architecture supports output_attentions under eager at all."
        )
    attn = out.attentions[layer_idx]                     # (B, heads, seq, seq)
    p = attn.clamp_min(1e-12)
    ent = -(p * p.log()).sum(dim=-1)                      # (B, heads, seq)
    mask = enc.attention_mask.bool().unsqueeze(1).expand_as(ent)   # broadcast over heads
    return float(ent[mask].mean().item())


def layerwise_cka_under_rotation_batched(model, tokenizer, texts, rotated_layer_idx, seed=0, device=DEVICE):
    """For each theta in the Part-4 sweep, collect every layer's hidden states (ONE batched,
    padded forward pass per theta, masked to real tokens before CKA) under the rotated
    condition and CKA against the theta=0 baseline, layer by layer."""
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    enc = tokenizer(list(texts), return_tensors='pt', padding=True).to(device)
    mask = enc.attention_mask.bool()

    def collect_all_layers(m):
        # output_hidden_states=True holds every layer's (B, T, H) activations on-GPU
        # simultaneously for the duration of this one forward call -- the memory-heavy part
        # of this probe. Each is moved to CPU/numpy and masked to real tokens immediately, and
        # `out` (plus the GPU-resident hidden_states tuple it holds) is deleted right after, so
        # nothing beyond that single forward pass's peak lingers once this function returns.
        with torch.inference_mode(), _amp_ctx():
            out = m(enc.input_ids, attention_mask=enc.attention_mask, output_hidden_states=True)
        acts = {li: hs[mask].float().cpu().numpy() for li, hs in enumerate(out.hidden_states)}
        del out
        return acts

    base_acts = collect_all_layers(model)
    free_memory()
    results = {}
    for theta, cond in rotation_angle_sweep_mlp(model, rotated_layer_idx,
                                                 [0.10, 0.25, 0.50, 0.75, 1.00], seed=seed, device=device):
        rot_acts = collect_all_layers(cond)
        results[theta] = {li: linear_cka(base_acts[li], rot_acts[li]) for li in base_acts}
        del rot_acts
        free_memory()   # ConditionModel `cond` restores the base model's weights on exit, but
                         # this still drops the per-theta CPU numpy arrays and any transient
                         # GPU buffers from that forward pass before the next theta starts
    return results


print("=" * 78)
print("POINT 4 FOLLOW-UP: LAYER-WISE CKA + ATTENTION ENTROPY UNDER THE ROTATION SWEEP")
print("=" * 78)
for name in ['Llama-3.2-1B', 'Qwen2.5-1.5B', 'Gemma-2-2b']:
    t0 = time.perf_counter()
    model, tok = models[name], tokenizers[name]
    print(f"\n-- {name} --")
    print_gpu_mem(f"before {name}")
    ent_base = attention_entropy_batched(model, tok, EVAL_CORPUS, TARGET_LAYER + 1)
    print(f"  baseline attention entropy (layer {TARGET_LAYER+1}): {ent_base:.3f} nats")
    cka_by_theta = layerwise_cka_under_rotation_batched(model, tok, EVAL_CORPUS, TARGET_LAYER)
    for theta, per_layer in cka_by_theta.items():
        downstream = np.mean([v for li, v in per_layer.items() if li > TARGET_LAYER])
        print(f"  theta={theta:.2f}: CKA at rotated layer {TARGET_LAYER} = {per_layer[TARGET_LAYER]:.4f}, "
              f"mean CKA over downstream layers = {downstream:.4f}")
    print(f"  ({name} Point 4 follow-up total: {time.perf_counter()-t0:.1f}s)")
    print_gpu_mem(f"after {name}")
    free_memory()

print("\nIf Llama's non-monotonicity is a representational phenomenon (redundant circuits /")
print("orbit-like near-returns), the CKA-vs-theta curve at the rotated layer, or immediately")
print("downstream, should also be non-monotonic, roughly tracking the accuracy dip/recovery.")
print("If CKA instead degrades monotonically while accuracy does not, the recovery is happening")
print("in how the (still-corrupted) representation is READ OUT, not in the representation itself.")


## Part 5 -- Mathematics: what's architecture-universal vs. architecture-specific

This section is analytical rather than another `run_battery` loop -- it states, for the
record, exactly which claims from Part 1-4 are theorems (true for any architecture with
the relevant linear structure) and which are empirical facts specific to these three
checkpoints.

**Theorem (permutation invariance of a linear-then-elementwise-then-linear block).**
Let $h = \sigma(x)$ be applied elementwise for any function $\sigma$ (ReLU, GELU, SiLU,
or a SwiGLU/GeGLU gate), and let $y = W_2\,\sigma(W_1 x + b_1)$. For any permutation
matrix $P$: $y = W_2 P^\top \, \sigma(P W_1 x + P b_1)$ for **any** elementwise $\sigma$,
because $P\sigma(z) = \sigma(Pz)$ whenever $\sigma$ is applied coordinatewise --
permutation commutes with *any* elementwise nonlinearity, not just ReLU. This is why the
aligned-permutation results above hold identically for GELU (GPT-2), SiLU (Llama/Qwen),
and GeGLU (Gemma) despite the differing activation functions -- confirmed empirically
above, but true as a matter of algebra, not something that needed three checkpoints to
discover.

**Contrast: scaling symmetry is NOT elementwise-nonlinearity-agnostic.** ReLU is
positive-homogeneous ($\mathrm{ReLU}(\alpha x) = \alpha\,\mathrm{ReLU}(x)$ for
$\alpha>0$), which supports an additional *rescaling* symmetry family
($W_1 \to \alpha W_1,\ W_2 \to W_2/\alpha$ for a diagonal positive $\alpha$) documented
in the mode-connectivity literature for ReLU networks specifically. **GELU, SiLU, and
GeGLU are not homogeneous** ($\sigma(\alpha x) \neq \alpha\,\sigma(x)$ in general), so
this rescaling symmetry does **not** hold exactly for any of the three model families
studied here (nor for GPT-2, which uses GELU) -- only the discrete permutation symmetry
survives across all four activation choices. This is a real mathematical distinction the
original GPT-2-only notebook could state but not contrast against a second family; here
it's directly checkable (see the code cell below).

**GQA-specific addition.** The permutation group acting exactly on attention is not
$S_{n_\text{heads}}$ (all head permutations) but the wreath-like subgroup that permutes
KV-head groups as blocks -- $S_{n_\text{kv}}$ acting on groups of size
$n_\text{heads}/n_\text{kv}$ -- as derived and empirically confirmed in Part 2's GQA
section. For $n_\text{kv} = n_\text{heads}$ (plain MHA, e.g. GPT-2) this collapses to
the familiar $S_{n_\text{heads}}$.

In [ ]:
print("=" * 78)
print("PART 5: NUMERIC CHECK -- SCALING SYMMETRY DOES NOT HOLD FOR SiLU/GeGLU/GELU")
print("=" * 78)

def check_scaling_symmetry(model, layer_idx, alpha=2.0, seed=0, device='cpu'):
    '''Applies W1 -> alpha*W1, W2 -> W2/alpha to gate_proj/down_proj and measures
    the logit deviation. For a homogeneous activation this would be exactly 0;
    for SiLU/GeGLU it should be nonzero, growing with |alpha - 1|. ConditionModel
    version -- only gate_proj/down_proj's tensors are duplicated.'''
    mlp = get_mlp(model, layer_idx)
    param_refs = [(mlp.gate_proj, 'weight'), (mlp.down_proj, 'weight')]
    variants = [mlp.gate_proj.weight.data * alpha, mlp.down_proj.weight.data / alpha]
    if mlp.gate_proj.bias is not None:
        param_refs.append((mlp.gate_proj, 'bias'))
        variants.append(mlp.gate_proj.bias.data * alpha)
    return ConditionModel(model, param_refs, variants, label=f'scaling_alpha{alpha}_L{layer_idx}')

for name in MODEL_CONFIGS:
    m, tok = models[name], tokenizers[name]
    sample_ids = tok(EVAL_CORPUS[0], return_tensors='pt').input_ids.to(DEVICE)
    with torch.no_grad():
        base_logits = m(sample_ids).logits
    for alpha in [1.01, 1.1, 1.5, 2.0]:
        scaled_m = check_scaling_symmetry(m, TARGET_LAYER, alpha=alpha, device=DEVICE)
        with torch.no_grad():
            diff = (base_logits - scaled_m(sample_ids).logits).abs().max().item()
        print(f"{name:>16s}  alpha={alpha:.2f}: max|logit diff| = {diff:.4f}  "
              f"(nonzero and growing with |alpha-1| confirms non-homogeneity)")


## Part 6 -- Unification: what carries over from GPT-2 and what's new

| Claim | GPT-2 (original notebook) | Qwen2.5-1.5B / Gemma-2-2b / Llama-3.2-1B (this notebook) |
|---|---|---|
| Neuron permutation of the MLP hidden axis is an exact functional symmetry | Confirmed (single `c_fc` projection) | Re-derived for the **two-branch** SwiGLU/GeGLU case -- same conclusion, different algebra; a **new** partial-symmetry-breaking mode (gate-only permutation) exists here that has no GPT-2 analogue |
| Uncompensated permutation collapses behavior | Confirmed | Confirmed, plus a **second**, GQA-specific collapse mode (within-group query-head reassignment) |
| Attention head permutation is exact when jointly applied | Confirmed for plain MHA ($n_\text{heads}=n_\text{kv}$) | Generalizes to $S_{n_\text{kv}}$ acting on **head groups**, not individual heads, whenever GQA is used ($n_\text{kv} < n_\text{heads}$) |
| Permutation commutes with any elementwise nonlinearity | Implicit (GELU only) | Explicitly checked across GELU/SiLU/GeGLU -- confirms the algebraic argument is nonlinearity-agnostic |
| ReLU-style rescaling symmetry | Not applicable (GPT-2 uses GELU) | Explicitly shown **not to hold** for SiLU/GeGLU either -- extends the negative result to two more non-homogeneous activations |
| Global residual-stream rotation is exact | Exact for GPT-2's LayerNorm-normalized residual stream (up to the learned affine, which the original notebook's `make_global_rotation` leaves untouched) | Expected to be **only approximate** for RMSNorm-normalized models, since RMSNorm's learned per-channel scale doesn't commute with a generic rotation -- flagged as a genuine finding to confirm from the Part 2 numeric output, not assumed |
| Exactness independent of model scale | Only testable within one architecture family (GPT-2 small/medium/large are scaled copies of the same design) | Testable across three **architecturally distinct** 1-3B models, a stronger genericity check |

**Bottom line.** The core theorem (Hecht-Nielsen 1990: relabeling hidden units is a
functional no-op) is architecture-universal and holds, as expected, across all three
families once each family's specific weight layout is correctly accounted for. What
changes across architectures is not *whether* the symmetry exists but *what its exact
group is* -- plain permutation for single-branch MLPs and MHA, group-permutation for
GQA, and a two-branch-consistent permutation for SwiGLU/GeGLU -- which is exactly the
kind of generalization question the original notebook's Part 3-6 headers (scaling,
cross-arch, cross-model, unification) were set up to ask, now answered with real
architectural diversity instead of same-architecture scale variants.

# References and citation map

This notebook studies functional identity under weight-space symmetries in real open-weight transformer checkpoints. The references below are the primary literature supporting the mathematical and methodological choices. Claims made by this project itself are explicitly labeled as project findings rather than attributed to prior work.

## Core identity / permutation-symmetry literature

1. **Hecht-Nielsen, R. (1990).** On the Algebraic Structure of Feedforward Network Weight Spaces. In *Advanced Neural Computers*, pp. 129–135.  
   https://doi.org/10.1016/B978-0-444-88400-8.50019-4  
   *Used for:* the foundational observation that hidden-unit permutations can leave the input-output function unchanged when adjacent weights are transformed consistently.

2. **Entezari, R., Sedghi, H., Saukh, O., & Neyshabur, B. (2022).** The Role of Permutation Invariance in Linear Mode Connectivity of Neural Networks. *ICLR 2022*.  
   https://arxiv.org/abs/2110.06296  
   *Used for:* permutation symmetries as function-preserving transformations and their relationship to weight-space geometry / mode connectivity.

3. **Ainsworth, S. K., Hayase, J., & Srinivasa, S. (2023).** Git Re-Basin: Merging Models Modulo Permutation Symmetries. *ICLR 2023*.  
   https://arxiv.org/abs/2209.04836  
   *Used for:* practical weight matching and alignment modulo permutation symmetries.

4. **Jordan, K., Sedghi, H., Saukh, O., Entezari, R., & Neyshabur, B. (2023).** REPAIR: Renormalizing Permuted Activations for Interpolation Repair. *ICLR 2023*.  
   https://arxiv.org/abs/2211.08429  
   *Used for:* the broader literature on permutation-induced representation mismatch and repair.

## Representation similarity

5. **Kornblith, S., Norouzi, M., Lee, H., & Hinton, G. (2019).** Similarity of Neural Network Representations Revisited. *ICML 2019*, 3519–3529.  
   https://proceedings.mlr.press/v97/kornblith19a.html  
   *Used for:* linear CKA and representation-similarity analysis.

6. **Raghu, M., Gilmer, J., Yosinski, J., & Sohl-Dickstein, J. (2017).** SVCCA: Singular Vector Canonical Correlation Analysis for Deep Learning Dynamics and Interpretability. *NeurIPS 2017*.  
   https://arxiv.org/abs/1706.05806  
   *Used for:* contextual background on comparing internal representations.

7. **Schönemann, P. H. (1966).** A Generalized Solution of the Orthogonal Procrustes Problem. *Psychometrika*, 31, 1–10.  
   https://doi.org/10.1007/BF02289451  
   *Used for:* Procrustes alignment / representation geometry.

## Causal intervention and transformer mechanisms

8. **Meng, K., Bau, D., Andonian, A., & Belinkov, Y. (2022).** Locating and Editing Factual Associations in GPT. *NeurIPS 2022*.  
   https://proceedings.neurips.cc/paper_files/paper/2022/hash/6f1d43d5a82a37e89b0665b33bf3a182-Abstract-Conference.html  
   *Used for:* the ROME-style causal tracing / intervention framing adapted in the causal-analysis section.

9. **Vig, J., Gehrmann, S., Belinkov, Y., et al. (2020).** Investigating Gender Bias in Language Models Using Causal Mediation Analysis. *NeurIPS 2020*.  
   https://arxiv.org/abs/2004.12265  
   *Used for:* causal mediation / activation intervention methodology.

10. **Olsson, C., Elhage, N., Nanda, N., et al. (2022).** In-context Learning and Induction Heads. Transformer Circuits Thread.  
    https://arxiv.org/abs/2209.11895  
    *Used for:* transformer circuit / attention-head background relevant to the attention-permutation experiments.

## Model and training-time context

11. **Biderman, S., Schoelkopf, H., Anthony, Q. G., et al. (2023).** Pythia: A Suite for Analyzing Large Language Models Across Training and Scaling. *ICML 2023*.  
    https://proceedings.mlr.press/v202/biderman23a.html  
    *Used for:* motivation for treating training checkpoints as a first-class object when studying representation development. The present notebook's three-model cross-family study is distinct from Pythia's controlled checkpoint suite.

12. **Radford, A., Wu, J., Child, R., et al. (2019).** Language Models are Unsupervised Multitask Learners. OpenAI.  
    https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf  
    *Used for:* GPT-2 architectural/model background inherited from the original project.

13. **Qwen/Qwen2.5-1.5B model card.**  
    https://huggingface.co/Qwen/Qwen2.5-1.5B  
    *Used for:* checkpoint identity and model configuration.

14. **Google Gemma 2 2B model card.**  
    https://huggingface.co/google/gemma-2-2b  
    *Used for:* checkpoint identity and model configuration.

15. **Meta Llama 3.2 1B model card.**  
    https://huggingface.co/meta-llama/Llama-3.2-1B  
    *Used for:* checkpoint identity and model configuration.

## Numerical / statistical background

16. **Higham, N. J. (2002).** *Accuracy and Stability of Numerical Algorithms* (2nd ed.). SIAM.  
    *Used for:* interpreting floating-point numerical error and why bf16 results should not be treated as symbolic exactness.

17. **Hendrycks, D., & Dietterich, T. (2019).** Benchmarking Neural Network Robustness to Common Corruptions and Perturbations. *ICLR 2019*.  
    https://arxiv.org/abs/1903.12261  
    *Used as general methodological context for controlled perturbation sweeps.

18. **Kuhn, H. W. (1955).** The Hungarian Method for the Assignment Problem. *Naval Research Logistics Quarterly*, 2, 83–97.  
    *Used for the assignment / matching background in cross-model alignment.

---
## Part 7 (added) -- Discussion: what kind of identity survives?


### From the numbers to the question

Every result above is, on its face, a statement about behavior under a specific algebraic
operation on a specific set of weights: permute the d_ff neurons and compensate the matching
rows/columns, and next-token accuracy doesn't move (0.4650 -> 0.4650 for Qwen, 0.4655 -> 0.4655
for Gemma, 0.4292 -> 0.4292 for Llama). Break that alignment, or replace the permutation with a
rotation, or delete a neuron outright, and accuracy moves by an amount that depends heavily on
*which* operation, *which* layer, and *which* model -- from "not at all" (Gemma under full local
rotation) to "almost entirely" (Qwen: a 97.5% relative drop, per Point 5's table above) to
"catastrophically, then partially, not fully" (Llama: dip to 0.0114 at theta=0.10, climbing back
to only 0.2522 -- 58.8% of baseline -- by theta=1.00, and never returning further).

"The network is the same identity, just reorganized" is defensible with total confidence about
aligned permutation, because the proof this notebook's Part 5/6 math sections cite shows the
output is *exactly* invariant, not just empirically close. "The network has become a different
function" is equally defensible for Qwen under rotation. But Llama's non-monotonic curve, and
Gemma's near-total invariance to an operation that *does* break Qwen, don't fit neatly into either
bucket. Those two results are the actual empirical content a functional-identity framing is trying
to describe -- worth stating plainly that arriving at that framing *from* these two results, rather
than importing it and reading the results as illustrations of it, is the order this notebook tries
to follow.


### Answering the operating question, piece by piece

The results assembled above answer pieces of the question stated at the top of this notebook
directly:

- **Which transformations preserve the function exactly.** Only a genuine symmetry of the
  parameterization -- an aligned permutation of a hidden-layer basis, or a simultaneously-permuted
  attention head (GQA-aware, per Part 2 above). Nothing else tested here qualifies.
- **Which properties of the transformation predict approximate vs. total loss.** Norm-preservation
  is not sufficient -- the rotation tested throughout is exactly orthogonal, and still destroys
  Qwen's function almost completely. What predicts survival, per the toy experiments in the Point
  3 deep-dive above, is closer to how large the perturbed component's contribution is *relative to*
  whatever it's being added into -- a claim about redundancy/dilution in the network's own internal
  accounting, not about the transformation in isolation.
- **Which architectural properties predict where the boundary sits.** The one candidate mechanism
  this notebook could actually test in isolation (the Point-3 toy experiments) points at
  normalization placement and residual-stream scale, not at activation function, GQA ratio, or
  RoPE -- Gemma-2's documented pre+post-norm sandwich and larger embedding scale are the more
  mechanistically apt explanation for its rotation-invariance than any of the alternatives that
  were tested and showed no effect.


### Where the philosophical vocabulary earns its place

Three ideas from outside machine learning map onto specific pieces of the above closely enough
that naming them adds precision rather than decoration -- each tied to a specific number already
on the page, not asserted independently of it.

**Functionalism and substrate-independence (Putnam, 1967; Block, 1978).** The aligned-permutation
results are close to a formal, checkable instance of the classical functionalist claim that what
makes a state the state it is, is its causal/functional role, not its physical encoding: two
parameter settings that differ in every entry of `gate_proj`/`up_proj` and their permuted
counterparts still implement *exactly* the same function, because the causal role each neuron
plays is preserved even though its address in the weight tensor changes. This is also why exact
functionalism is the wrong frame on its own: the interesting results here (Gemma's
rotation-invariance, Llama's partial recovery) are cases where the causal role is *partially*
disrupted and the function *partially* persists, which exact functionalism has no native
vocabulary for -- that gap is the actual empirical territory the Point 3/4 deep-dives are in.

**The Ship of Theseus, and which version of the puzzle applies.** The classical puzzle (via
Plutarch) asks whether an object with every part replaced remains "the same ship." Aligned
permutation is the *easy* version -- every entry changes, but nothing the ship *does* changes, so
there's no real tension. The harder, more relevant version -- not "replace every plank" but
"twist the planks without removing any" (the rotation sweep) -- is what Llama's curve actually
probes: not a monotonic decline as the twisting increases, but a dip and a partial, incomplete
recovery. That's closer to the actual shape of the puzzle than either "obviously the same" or
"obviously different."

**Degeneracy in biological neural systems (Edelman & Gally, 2001).** Degeneracy -- structurally
different elements performing the same function, explicitly distinguished from redundancy
(identical elements performing the same function) -- is a closer fit to Gemma's and Llama's
results than generic "robustness." Gemma's invariance under a rotation that is, by construction,
*not* a symmetry of its parameterization, together with Llama's non-monotonic partial recovery,
look like signatures of degenerate rather than merely redundant structure. Stated as a naming of
the pattern visible in the data -- one worth testing further with the layer-wise-CKA /
attention-entropy follow-up specified in the Point 4 deep-dive above -- not as something verified
at the level of individual circuits.


**A postscript on how much of this is settled vs. still open.** Experiment C above closes
one of the two gaps the reviewer's Point 4 named (redundancy vs. low rank) at the toy-model
level; the CKA/attention-entropy PENDING-GPU cell closes the other (whether the effect
shows up representationally on the real checkpoints, not just in a synthetic replica). What
this notebook does **not** yet have is either of those run on the real Qwen/Gemma/Llama
weights -- every "degeneracy" and "dilution" claim above is a toy-model-validated hypothesis
about the real checkpoints, not a measurement of them. That distinction is the difference
between "this is a plausible mechanism" and "this is the mechanism," and this notebook only
earns the first claim until the PENDING-GPU cells are actually run.


### What this does and doesn't license

None of the above claims that Qwen2.5, Gemma-2, or Llama-3.2 possess anything resembling
biological neuroplasticity, and none of it is a claim about consciousness or experience. The claim
on the table is narrower: **"functional identity" here names a graded, transformation- and
architecture-dependent property this notebook can measure** -- exact under provable symmetries;
approximately preserved when a perturbed component is diluted by a large residual stream, per the
toy experiments; non-monotonically and only partially preserved when neither of those holds, as in
Llama's rotation curve -- and the philosophical vocabulary above is useful exactly to the extent it
gives that graded property sharper edges: distinguishing "same by proof" from "same by dilution"
from "partially recovered by some mechanism this notebook has flagged but not yet isolated." Where
the mechanism isn't isolated -- Llama's non-monotonicity chief among them -- the honest next step
is the GPU-dependent layer-wise CKA / attention-entropy analysis specified above, not a narrative
that resolves the open question before the data that would resolve it has been collected.


### References added for this discussion

- Team, Gemma, et al. (2024). *Gemma 2: Improving Open Language Models at a Practical Size.*
  Google DeepMind. arXiv:2408.00118. (Source for the normalization-placement facts used in Point 3:
  RMSNorm applied to both the input and output of every attention/feedforward sub-layer, plus the
  family's larger embedding scale.)
- Elhage, N., et al. (2022). *Toy Models of Superposition.* Anthropic. (Motivates the
  redundant/degenerate-circuit hypothesis for Llama's non-monotonic curve, Point 4.)
- Putnam, H. (1967). Psychological Predicates. In *Art, Mind, and Religion.* (Functionalism /
  substrate-independence.)
- Block, N. (1978). Troubles with Functionalism. *Minnesota Studies in the Philosophy of Science.*
  (Standard caveats to functionalism, relevant to where the framing does and doesn't apply above.)
- Plutarch. *Life of Theseus*, ~75 CE (earliest surviving statement of the Ship of Theseus
  problem).
- Edelman, G. M., & Gally, J. A. (2001). Degeneracy and complexity in biological systems.
  *PNAS*, 98(24), 13763-13768. (Degeneracy vs. redundancy -- the closest existing biological
  vocabulary to the Gemma/Llama results above.)
